# AMP Evaluation Pipeline
Pipeline: physicochemical filters, novelty, AMPlify, toxicity, and LLAMP multi-organism scoring.


### Project paths

This notebook lives in `notebooks/`. Shared code is under `src/`, data under `data/`.
On Colab, set `ROOT` to your cloned repo (or Drive copy) instead of `Path('..')`.


In [ ]:
import sys
from pathlib import Path

# Repo root (parent of notebooks/). On Colab, replace with your clone path, e.g.:
# ROOT = Path('/content/drive/MyDrive/your-thesis-repo')
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.paths import DATA_DIR, CHECKPOINTS_DIR, RESULTS_DIR, PROJECT_ROOT
print('PROJECT_ROOT :', PROJECT_ROOT)
print('DATA_DIR     :', DATA_DIR)
print('CHECKPOINTS  :', CHECKPOINTS_DIR)
print('RESULTS      :', RESULTS_DIR)


## Environment setup

Mount Drive and install dependencies (BLAST, CD-HIT, AMPlify, ToxinPred3).


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip install biopython peptides pandas toxinpred3 scikit-learn==1.2.2 "numpy<2.0.0"
!pip install -q tensorflow accelerate safetensors
!apt-get install ncbi-blast+ cd-hit -y

import os
if not os.path.exists("/content/AMPlify"):
    !git clone https://github.com/bcgsc/AMPlify.git /content/AMPlify


## Phase 1 — Physicochemical gate

Length, charge, and hydrophobicity sanity check (pass/fail).


In [ ]:
import pandas as pd
import peptides

class Phase1_PhysicochemicalGate:
    def __init__(self, results: pd.DataFrame):
        self.results = results

    def evaluate(self) -> pd.DataFrame:
        print("Running Phase 1: Physicochemical Sanity Gate...")

        hydrophobic_aas = set("VILFMWAY")

        def gate_and_features(seq):
            default = pd.Series({
                "phase1_pass":               False,
                "phase1_length":            0.0,
                "phase1_charge":            0.0,
                "phase1_hydrophobic_frac":  0.0,
                "phase1_hydrophobic_moment": 0.0
            })

            if not (6 <= len(seq) <= 100):
                return default

            pep = peptides.Peptide(seq)
            charge = pep.charge(pH=7.4)

            if charge < -2:
                return default

            hydro_frac = sum(1 for aa in seq if aa in hydrophobic_aas) / len(seq)
            if not (0.1 <= hydro_frac <= 0.73):
                return default

            return pd.Series({
                "phase1_pass":               True,
                "phase1_length":             float(len(seq)),
                "phase1_charge":             round(charge, 2),
                "phase1_hydrophobic_frac":   round(hydro_frac, 3),
                "phase1_hydrophobic_moment": round(pep.hydrophobic_moment(), 4)
            })

        feature_cols = [
            "phase1_pass",
            "phase1_length",
            "phase1_charge",
            "phase1_hydrophobic_frac",
            "phase1_hydrophobic_moment"
        ]

        features = self.results["sequence"].apply(gate_and_features)

        for col in feature_cols:
            self.results[col] = features[col]

        n_passed = self.results["phase1_pass"].sum()
        print(f"  ✓ Phase 1 Complete: {n_passed}/{len(self.results)} sequences passed.")
        return self.results


## Phase 2 — BLAST reference database

Build a local BLAST DB from GRAMPA reference AMPs.


In [ ]:
import os
import pandas as pd
import subprocess

def setup_grampa_blast_database():
    db_fasta = "amp_reference_db.fasta"
    db_name = "amp_db"

    grampa_url = "https://raw.githubusercontent.com/zswitten/Antimicrobial-Peptides/master/data/grampa.csv"

    print("Fetching and building BLAST database from GRAMPA dataset...")

    try:
        df_grampa = pd.read_csv(grampa_url)

        unique_sequences = df_grampa['sequence'].dropna().unique()

        print(f"  ✓ Found {len(unique_sequences)} unique, verified reference AMPs.")

        with open(db_fasta, "w") as f:
            for i, seq in enumerate(unique_sequences):
                f.write(f">grampa_{i}\n{seq.upper()}\n")
        print("  ✓ Converted entries to 'amp_reference_db.fasta'.")

        cmd = ["makeblastdb", "-in", db_fasta, "-dbtype", "prot", "-out", db_name]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
        print(f"  ✓ Local BLAST database '{db_name}' compiled and ready.")

        return db_name

    except Exception as e:
        raise RuntimeError(f"Failed to process GRAMPA database: {str(e)}")

AMP_DB_PATH = setup_grampa_blast_database()


## Phase 2 — Novelty scorer

Score novelty from BLAST identity and Shannon entropy (0–100).


In [ ]:
import os
import math
import subprocess
from collections import Counter
import pandas as pd
from Bio.Blast import NCBIXML

class Phase2_NoveltyScorer:

    def __init__(self, results: pd.DataFrame, db_path: str = "amp_db"):
        self.results = results
        self.db_path = db_path

    def evaluate(
        self,
        cd_hit_identity: float = 0.95,
        min_entropy: float = 2.0,
        max_entropy: float = 5.0,
    ) -> pd.DataFrame:

        mask = self.results["phase1_pass"] == True
        candidates = self.results.loc[mask, "sequence"].copy()

        if candidates.empty:
            print("Phase 2: No sequences passed Phase 1. Skipping.")
            return self.results

        print("Running Phase 2: Novelty Scorer...")

        blast_scores  = self._run_blast(candidates)
        entropy_scores = candidates.apply(
            lambda seq: self._entropy_score(seq, min_entropy, max_entropy)
        )

        novelty = (blast_scores * entropy_scores).pow(0.5).round(2)

        self.results.loc[mask, "phase2_blast_identity"]  = blast_scores.apply(
            lambda s: round(100 - s, 2)
        )
        self.results.loc[mask, "phase2_entropy"]         = candidates.apply(
            lambda seq: round(self._shannon_entropy(seq), 4)
        )
        self.results.loc[mask, "phase2_score"]           = novelty

        print(f"    Phase 2 Complete: scored {mask.sum()} sequences.")
        print(f"    Mean novelty score : {novelty.mean():.1f}")
        print(f"    Sequences > 80     : {(novelty > 80).sum()}")
        print(f"    Sequences < 20     : {(novelty < 20).sum()}\n")

        return self.results

    def _run_blast(self, candidates: pd.Series) -> pd.Series:
        query_fasta = "_phase2_query.fasta"
        out_xml     = "_phase2_blast.xml"

        with open(query_fasta, "w") as f:
            for idx, seq in candidates.items():
                f.write(f">seq_{idx}\n{seq}\n")

        cmd = [
            "blastp",
            "-query",  query_fasta,
            "-db",     self.db_path,
            "-evalue", "10",
            "-outfmt", "5",
            "-out",    out_xml,
            "-task",   "blastp-short",
        ]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        identity_map = {idx: 0.0 for idx in candidates.index}

        if os.path.exists(out_xml):
            with open(out_xml) as h:
                for record in NCBIXML.parse(h):
                    idx = int(record.query.split("_")[1])
                    if record.alignments:
                        hsp = record.alignments[0].hsps[0]
                        identity_map[idx] = (hsp.identities / hsp.align_length) * 100.0

        for fp in [query_fasta, out_xml]:
            if os.path.exists(fp):
                os.remove(fp)

        def identity_to_score(identity_pct: float) -> float:
            return round(100.0 * (1.0 - identity_pct / 100.0) ** 2, 2)

        return pd.Series(
            {idx: identity_to_score(identity_map[idx]) for idx in candidates.index}
        )

    @staticmethod
    def _shannon_entropy(seq: str, n: int = 3) -> float:
        if len(seq) < n:
            return 0.0
        ngrams = [seq[i:i+n] for i in range(len(seq) - n + 1)]
        counts = Counter(ngrams)
        total  = len(ngrams)
        return -sum((c / total) * math.log2(c / total) for c in counts.values())

    def _entropy_score(self, seq: str, min_entropy: float, max_entropy: float) -> float:
        h = self._shannon_entropy(seq)
        if h <= min_entropy:
            return max(0.0, round((h / min_entropy) * 10.0, 2))
        score = (h - min_entropy) / (max_entropy - min_entropy) * 100.0
        return round(min(score, 100.0), 2)


## Phase 3 — AMPlify scorer

Antimicrobial activity score from the AMPlify neural network.


In [ ]:
import os
import sys
import glob
import tempfile
import subprocess
import numpy as np
import pandas as pd

class Phase3_AMPlifyScorer:

    def __init__(self, results: pd.DataFrame, amplify_path: str = "/content/AMPlify/src/AMPlify.py"):
        self.results      = results
        self.amplify_path = amplify_path

        if "sequence" not in self.results.columns:
            raise ValueError("Master DataFrame must contain a 'sequence' column.")

        self._apply_keras_compatibility_patches()

    def evaluate(self) -> pd.DataFrame:
        mask = self.results["phase1_pass"] == True
        candidates = self.results.loc[mask, "sequence"]

        if candidates.empty:
            print("Phase 3: No sequences passed Phase 1. Skipping.")
            return self.results

        print("Running Phase 3: AMPlify Deep Neural Network Scorer...")

        with tempfile.TemporaryDirectory() as tmpdir:
            fasta_path = os.path.join(tmpdir, "input_seqs.fasta")
            out_dir    = os.path.join(tmpdir, "amplify_out")
            os.makedirs(out_dir, exist_ok=True)

            self._write_temp_fasta(candidates, fasta_path)

            cmd = [
                sys.executable, self.amplify_path,
                "-s",  fasta_path,
                "-od", out_dir,
                "-of", "tsv",
            ]

            try:
                subprocess.run(cmd, capture_output=True, text=True, check=True)
            except subprocess.CalledProcessError as e:
                print(f"AMPlify Execution Failed:\n{e.stderr}")
                raise RuntimeError("AMPlify model failed to process the sequences.")

            tsv_files = glob.glob(os.path.join(out_dir, "*.tsv"))
            if not tsv_files:
                raise FileNotFoundError("AMPlify ran but produced no output TSV.")

            df_out = pd.read_csv(tsv_files[0], sep="\t")

        scored = 0
        for _, row in df_out.iterrows():
            original_idx = int(str(row["Sequence_ID"]).split("_")[1])
            if original_idx not in self.results.index:
                continue
            raw_score  = float(row["AMPlify_log_scaled_score"])
            phase3_score = self._amplify_to_score(raw_score)
            self.results.at[original_idx, "phase3_score"] = phase3_score
            scored += 1

        print(f"    Phase 3 Complete: {scored}/{mask.sum()} sequences scored by AMPlify.")
        print(f"    Mean phase3_score : {self.results.loc[mask, 'phase3_score'].mean():.1f}\n")

        return self.results

    @staticmethod
    def _amplify_to_score(log_scaled_score: float) -> float:
        if pd.isna(log_scaled_score):
            return 0.0
        prob = 1.0 / (1.0 + np.exp(-0.1 * (log_scaled_score - 15)))
        return round(prob * 100.0, 2)

    @staticmethod
    def _write_temp_fasta(candidates: pd.Series, path: str):
        with open(path, "w") as fh:
            for idx, seq in candidates.items():
                fh.write(f">seq_{idx}\n{seq}\n")

    def _apply_keras_compatibility_patches(self):
        base_dir   = os.path.dirname(os.path.dirname(self.amplify_path))
        layers_path = os.path.join(base_dir, "src", "layers.py")

        if not os.path.exists(layers_path):
            print(f"Could not find layers.py at {layers_path}. Skipping patch.")
            return

        with open(layers_path, "r") as f:
            src = f.read()

        if "K.batch_dot(" not in src and "K.int_shape(" not in src:
            return

        print("Applying TensorFlow / Keras compatibility patches to AMPlify layers...")

        helper_functions = """
def _batch_dot(x, y, axes=None):
    if axes is not None:
        if isinstance(axes, int):
            axes = [axes, axes]
        if axes[1] == 2:
            y = tf.transpose(y, perm=[0, 2, 1])
        elif axes[0] == 1:
            x = tf.transpose(x, perm=[0, 2, 1])
    return tf.matmul(x, y)

def _int_shape(x):
    return tuple(x.shape.as_list())
"""

        if "import tensorflow as tf" not in src:
            src = "import tensorflow as tf\n" + helper_functions + src
        else:
            src = src.replace(
                "import tensorflow as tf",
                "import tensorflow as tf\n" + helper_functions,
                1,
            )

        replacements = {
            "K.batch_dot("         : "_batch_dot(",
            "K.int_shape("         : "_int_shape(",
            "K.permute_dimensions(": "tf.transpose(",
            "K.tile("              : "tf.tile(",
            "K.maximum("           : "tf.maximum(",
            "K.arange("            : "tf.range(",
            "K.dot("               : "tf.linalg.matmul(",
            "K.shape("             : "tf.shape(",
            "K.reshape("           : "tf.reshape(",
            "K.transpose("         : "tf.transpose(",
            "K.expand_dims("       : "tf.expand_dims(",
            "K.squeeze("           : "tf.squeeze(",
            "K.concatenate("       : "tf.concat(",
            "K.softmax("           : "tf.nn.softmax(",
            "K.tanh("              : "tf.math.tanh(",
            "K.relu("              : "tf.nn.relu(",
            "K.cast("              : "tf.cast(",
            "K.sum("               : "tf.reduce_sum(",
            "K.mean("              : "tf.reduce_mean(",
            "K.max("               : "tf.reduce_max(",
            "K.min("               : "tf.reduce_min(",
            "K.zeros_like("        : "tf.zeros_like(",
            "K.ones_like("         : "tf.ones_like(",
            "K.zeros("             : "tf.zeros(",
            "K.ones("              : "tf.ones(",
            "K.abs("               : "tf.abs(",
            "K.exp("               : "tf.exp(",
            "K.log("               : "tf.math.log(",
            "K.sqrt("              : "tf.sqrt(",
            "K.clip("              : "tf.clip_by_value(",
            "K.epsilon("           : "tf.keras.backend.epsilon(",
            "K.floatx("            : "tf.keras.backend.floatx(",
            "K.variable("          : "tf.Variable(",
            "K.constant("          : "tf.constant(",
        }

        for old, new in replacements.items():
            src = src.replace(old, new)

        src = src.replace(
            "tf.expand_dims(self.context)",
            "tf.expand_dims(self.context, axis=-1)",
        )

        with open(layers_path, "w") as f:
            f.write(src)

        print("Patches applied successfully.")


## Phase 4 — Toxicity scorer

Safety score from ToxinPred3 (higher = safer).


In [ ]:
import os
import tempfile
import numpy as np
import pandas as pd

import toxinpred3.python_scripts.toxinpred3 as tp

class Phase4_ToxicityScorer:

    def __init__(self, results: pd.DataFrame):
        self.results = results

        if "sequence" not in self.results.columns:
            raise ValueError("Master DataFrame must contain a 'sequence' column.")

    def evaluate(self) -> pd.DataFrame:
        mask = self.results["phase1_pass"] == True
        candidates = self.results.loc[mask, "sequence"]

        if candidates.empty:
            print("Phase 4: No sequences passed Phase 1. Skipping.")
            return self.results

        print("Running Phase 4: ToxinPred3 Toxicity Scorer...")

        original_cwd = os.getcwd()

        try:
            with tempfile.TemporaryDirectory() as tmpdir:
                os.chdir(tmpdir)

                fasta_path = os.path.join(tmpdir, "input.fasta")
                with open(fasta_path, "w") as f:
                    for idx, seq in candidates.items():
                        f.write(f">seq_{idx}\n{seq}\n")

                seq_list = list(candidates)

                tp.aac_comp(seq_list, "seq.aac")
                tp.dpc_comp(seq_list, "seq.dpc")

                os.system("sed -i 's/,$//g' seq.aac")
                os.system("sed -i 's/,$//g' seq.dpc")

                model_path = os.path.join(
                    os.path.dirname(tp.__file__),
                    "../model/toxinpred3.0_model.pkl"
                )

                original_loadtxt = np.loadtxt
                def safe_loadtxt(*args, **kwargs):
                    kwargs['ndmin'] = 2
                    return original_loadtxt(*args, **kwargs)

                np.loadtxt = safe_loadtxt

                try:
                    tp.prediction("seq.aac", "seq.dpc", model_path, "seq.pred")
                finally:
                    np.loadtxt = original_loadtxt

                tp.class_assignment("seq.pred", 0.38, "seq.out")

                if not os.path.exists("seq.out"):
                    print("  ToxinPred3 produced no output file.")
                    return self.results

                df_out = pd.read_csv("seq.out").fillna(0)

                candidate_indices = list(candidates.index)

                for i, row in df_out.iterrows():
                    if i >= len(candidate_indices):
                        break
                    original_idx = candidate_indices[i]
                    ml_score     = float(row.get("ML Score", 0))

                    phase4_score = round((1.0 - ml_score) * 100.0, 2)
                    self.results.at[original_idx, "phase4_score"] = phase4_score

        finally:
            os.chdir(original_cwd)

        scored = (self.results.loc[mask, "phase4_score"] > 0).sum()
        n_toxic = (self.results.loc[mask, "phase4_score"] < 62).sum()
        print(f" Phase 4 Complete: {scored}/{mask.sum()} sequences scored.")
        print(f" Predicted toxic (phase4_score < 62): {n_toxic}\n")

        return self.results


## LLAMP setup

Load LLAMP weights, peptide-tuned ESM-2, and genome features.


In [ ]:
import os, sys, math, difflib, importlib.util
import peptides
import numpy as np
import torch
import pandas as pd
from transformers import AutoTokenizer
import gdown

DRIVE_PATH     = "/content/drive/MyDrive/DBAASP_Bioinformatics"
LLAMP_WEIGHTS  = f"{DRIVE_PATH}/LLAMP_weights/llamp_weights.pt"
ESM_DIR        = f"{DRIVE_PATH}/LLAMP_weights/peptide_tuned_ESM-2"
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(ESM_DIR, exist_ok=True)

if not os.path.exists("/content/LLAMP"):
    os.system("git clone https://github.com/GIST-CSBL/LLAMP.git /content/LLAMP")

for p in ["/content/LLAMP", "/content/LLAMP/utils"]:
    if p not in sys.path: sys.path.append(p)

if not os.path.exists(LLAMP_WEIGHTS):
    print("Downloading LLAMP weights...")
    gdown.download(id="1hmfL7uRZsHo4pn0o0nqaqPcntxGJIhE7",
                   output=LLAMP_WEIGHTS, quiet=False)

if not os.path.exists(f"{ESM_DIR}/model.safetensors"):
    print("Downloading ESM-2 weights...")
    from transformers import AutoModel
    AutoTokenizer.from_pretrained("Daehun/peptide_tuned_ESM-2").save_pretrained(ESM_DIR)
    AutoModel.from_pretrained("Daehun/peptide_tuned_ESM-2").save_pretrained(ESM_DIR)

spec = importlib.util.spec_from_file_location(
    "llamp_model", "/content/LLAMP/utils/model.py"
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

llamp_model = mod.LLAMP(pretrained_model=ESM_DIR, pooling="mean")
llamp_model.load_state_dict(
    torch.load(LLAMP_WEIGHTS, map_location=DEVICE), strict=False
)
llamp_model.to(DEVICE).eval()
llamp_tokenizer = AutoTokenizer.from_pretrained(ESM_DIR)

genomic_data = torch.load(
    "/content/LLAMP/data/Genomic_featrues/genome_features.pt",
    map_location="cpu", weights_only=False
)
AVAILABLE_ORGANISMS = sorted(genomic_data.keys())
print(f"LLAMP ready on {DEVICE}. {len(AVAILABLE_ORGANISMS)} organisms available.")

def _get_genome_tensor(organism_name, target_dim=340):
    matches = difflib.get_close_matches(
        organism_name, AVAILABLE_ORGANISMS, n=1, cutoff=0.3
    )
    if not matches:
        raise ValueError(
            f"Organism '{organism_name}' not found. "
            f"Try one of: {AVAILABLE_ORGANISMS[:5]}"
        )
    matched = matches[0]
    raw     = genomic_data[matched]
    if isinstance(raw, list) and len(raw) == 1: raw = raw[0]
    if hasattr(raw, "ndim") and raw.ndim > 1:   raw = raw[0]
    if hasattr(raw, "tolist"): raw = raw.tolist()
    nums = []
    for item in (raw if isinstance(raw, (list, np.ndarray)) else [raw]):
        for sub in (item if isinstance(item, (list, np.ndarray)) else [item]):
            if isinstance(sub, (int, float, np.number)):
                nums.append(float(sub))
    t = torch.tensor(nums).float().unsqueeze(0)
    if   t.shape[-1] > target_dim: t = t[:, :target_dim]
    elif t.shape[-1] < target_dim: t = torch.nn.functional.pad(
        t, (0, target_dim - t.shape[-1])
    )
    return t.to(DEVICE), matched

def _predict_mic(sequence, genome_tensor):
    inputs = llamp_tokenizer(
        sequence, return_tensors="pt",
        padding=True, truncation=True, max_length=512
    )
    with torch.no_grad():
        log10_mic = llamp_model(
            inputs["input_ids"].to(DEVICE),
            inputs["attention_mask"].to(DEVICE),
            genome_tensor
        )

    mic_uM = 10 ** float(log10_mic.item())
    mw = peptides.Peptide(sequence).molecular_weight()
    mic_ug_ml = round(mic_uM * mw / 1000.0, 4)

    log_mic = math.log2(max(mic_ug_ml, 0.01))
    log_lo, log_hi = math.log2(32), math.log2(256)
    score = round(max(0.0, min(100.0, 100.0 * (1.0 - (log_mic - log_lo) / (log_hi - log_lo)))), 2)

    return mic_ug_ml, score


## LLAMP helpers

Sequence loading, Phases 1–4 runner, and memory-safe LLAMP MIC prediction.


In [ ]:
import os, re
import pandas as pd

VALID_AAS = set("ACDEFGHIKLMNPQRSTVWY")

def _is_valid_sequence(s):
    s = s.strip().upper()
    return len(s) >= 4 and all(c in VALID_AAS for c in s)

def _load_sequences_from_file(path):
    ext = os.path.splitext(path)[1].lower()

    if ext in (".fasta", ".fa"):
        seqs = []
        with open(path) as f:
            buf = []
            for line in f:
                line = line.strip()
                if not line: continue
                if line.startswith(">"):
                    if buf: seqs.append(("".join(buf).upper(), None))
                    buf = []
                else:
                    buf.append(line)
            if buf: seqs.append(("".join(buf).upper(), None))
        return seqs

    if ext == ".txt":
        seqs = []
        with open(path) as f:
            for line in f:
                s = line.strip().upper()
                if s and _is_valid_sequence(s):
                    seqs.append((s, None))
        return seqs

    if ext == ".csv":
        df = pd.read_csv(path)
        df.columns = [c.strip().lower() for c in df.columns]
        if "sequence" not in df.columns:
            raise ValueError(f"CSV must have a 'sequence' column. Found: {list(df.columns)}")
        org_col = "organism" if "organism" in df.columns else None
        seqs = []
        for _, row in df.iterrows():
            seq = str(row["sequence"]).strip().upper()
            org = str(row[org_col]).strip() if org_col else None
            if _is_valid_sequence(seq):
                seqs.append((seq, org))
        return seqs

    raise ValueError(f"Unsupported file type: {ext}. Use .fasta, .txt, or .csv")

def _run_phases_1_to_4(sequences):
    df = pd.DataFrame({"sequence": sequences})

    p1 = Phase1_PhysicochemicalGate(df.copy())
    df = p1.evaluate()
    p2 = Phase2_NoveltyScorer(df, db_path=AMP_DB_PATH)
    df = p2.evaluate()
    p3 = Phase3_AMPlifyScorer(df)
    df = p3.evaluate()
    p4 = Phase4_ToxicityScorer(df)
    df = p4.evaluate()

    return df

import gc

CLEANUP_EVERY = 200

def _run_llamp(sequence_organism_pairs):
    rows = []

    from collections import defaultdict
    by_org = defaultdict(list)
    for seq, org in sequence_organism_pairs:
        by_org[org].append(seq)

    for org, seqs in by_org.items():
        try:
            genome_tensor, matched = _get_genome_tensor(org)
            print(f"  LLAMP organism: '{matched}' ({len(seqs)} sequences)")
        except ValueError as e:
            print(f"  Skipping '{org}': {e}")
            for seq in seqs:
                rows.append({
                    "sequence": seq, "organism": org,
                    "llamp_mic_ug_ml": None, "llamp_score": None
                })
            continue

        n = len(seqs)
        for i, seq in enumerate(seqs):
            try:
                mic, score = _predict_mic(seq, genome_tensor)
            except Exception as e:
                print(f"  Warning: failed on '{seq[:20]}': {e}")
                mic, score = None, None

            rows.append({
                "sequence": seq,
                "organism": matched,
                "llamp_mic_ug_ml": mic,
                "llamp_score": score,
            })

            if (i + 1) % CLEANUP_EVERY == 0:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                print(
                    f"    ...{i + 1}/{n} sequences done for '{matched}' "
                    f"(memory cleanup checkpoint)"
                )

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        del genome_tensor

    return pd.DataFrame(rows)

def _predict_mic(sequence, genome_tensor):
    inputs = llamp_tokenizer(
        sequence, return_tensors="pt",
        padding=True, truncation=True, max_length=512
    )
    input_ids = inputs["input_ids"].to(DEVICE)
    attention_mask = inputs["attention_mask"].to(DEVICE)

    with torch.no_grad():
        log10_mic = llamp_model(input_ids, attention_mask, genome_tensor)
        mic_value = float(log10_mic.item())

    del inputs, input_ids, attention_mask, log10_mic

    mic_uM = 10 ** mic_value
    mw = peptides.Peptide(sequence).molecular_weight()
    mic_ug_ml = round(mic_uM * mw / 1000.0, 4)

    log_mic = math.log2(max(mic_ug_ml, 0.01))
    log_lo, log_hi = math.log2(32), math.log2(256)
    score = round(
        max(0.0, min(100.0, 100.0 * (1.0 - (log_mic - log_lo) / (log_hi - log_lo)))),
        2,
    )

    return mic_ug_ml, score

def evaluate(input_data, organism=None, save_dir=None):

    is_file   = False
    file_path = None

    if isinstance(input_data, str) and os.path.exists(input_data):
        is_file   = True
        file_path = input_data
        pairs     = _load_sequences_from_file(file_path)
    elif isinstance(input_data, str) and _is_valid_sequence(input_data):
        pairs = [(input_data.strip().upper(), organism)]
    elif isinstance(input_data, list):
        pairs = [(s.strip().upper(), organism) for s in input_data if _is_valid_sequence(s)]
    else:
        raise ValueError(
            "input_data must be a peptide sequence string, a list of sequences, "
            "or a path to a .fasta / .txt / .csv file."
        )

    if not pairs:
        print("No valid sequences found in input.")
        return pd.DataFrame()

    has_organisms = any(org is not None for _, org in pairs)
    mode = 2 if (has_organisms or organism is not None) else 1

    print(f"Input: {len(pairs)} sequence(s)  |  Mode: {'LLAMP (organism-specific)' if mode == 2 else 'Phases 1-4'}")

    if mode == 1:
        sequences = [seq for seq, _ in pairs]
        result_df = _run_phases_1_to_4(sequences)

        output_cols = [
            "sequence",
            "phase1_pass",
            "phase2_score",
            "phase3_score",
            "phase4_score",
        ]
        output_cols = [c for c in output_cols if c in result_df.columns]
        result_df   = result_df[output_cols]

    else:
        if organism is not None and not has_organisms:
            pairs = [(seq, organism) for seq, _ in pairs]

        result_df = _run_llamp(pairs)
        result_df = result_df[["sequence", "organism", "llamp_mic_ug_ml", "llamp_score"]]
        result_df = result_df.sort_values("llamp_score", ascending=False, na_position="last")
        result_df = result_df.reset_index(drop=True)

    if is_file or save_dir is not None:
        if save_dir is None:
            save_dir = os.path.dirname(os.path.abspath(file_path)) if file_path else DRIVE_PATH
        os.makedirs(save_dir, exist_ok=True)

        base = os.path.splitext(os.path.basename(file_path))[0] if file_path else "input"
        suffix = "llamp" if mode == 2 else "evaluation"
        out_path = os.path.join(save_dir, f"{base}_{suffix}.csv")

        result_df.to_csv(out_path, index=False)
        print(f"\nSaved → {out_path}")

    print(f"\nResults ({len(result_df)} sequences):")
    if mode == 1:
        n_pass = result_df["phase1_pass"].sum() if "phase1_pass" in result_df.columns else "?"
        print(f"  Phase 1 pass: {n_pass}/{len(result_df)}")
        for col in ["phase2_score", "phase3_score", "phase4_score"]:
            if col in result_df.columns:
                vals = result_df[col].dropna()
                if len(vals):
                    print(f"  {col}: mean={vals.mean():.1f}  min={vals.min():.1f}  max={vals.max():.1f}")
    else:
        vals = result_df["llamp_score"].dropna()
        if len(vals):
            print(f"  LLAMP score: mean={vals.mean():.1f}  min={vals.min():.1f}  max={vals.max():.1f}")
            print(f"  Predicted MIC range: {result_df['llamp_mic_ug_ml'].min():.2f} – {result_df['llamp_mic_ug_ml'].max():.2f} µg/mL")
        print(f"\n  Top 5:")
        print(result_df.head(5).to_string(index=False))

    return result_df


## Universal evaluation helpers

Shared W&B logging, dashboards, phase views, and LLAMP cross-organism utilities.


In [ ]:
!pip install -q wandb openpyxl matplotlib seaborn

import gc, io, json, math, os, textwrap
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.pyplot.ioff()
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import wandb

try:
    from PIL import Image as PILImage
except ImportError:
    !pip install -q pillow
    from PIL import Image as PILImage

OUTPUT_DIR          = "/content/drive/MyDrive/DBAASP_Bioinformatics/evaluation"
WANDB_ENTITY        = "AMP_Generation"
WANDB_PROJECT       = "final_evaluation"
WANDB_RUN_NAME      = None

PHASE2_THRESHOLD = 60.0
PHASE3_THRESHOLD = 65.0
PHASE4_THRESHOLD = 62.0
AMPLIFY_PATH     = "/content/AMPlify/src/AMPlify.py"

RUN_PHASES       = True
RUN_LLAMP        = True

# Target/home organism for this peptide set
HOME_ORGANISM    = "enterica"

# Organisms LLAMP scores against
LLAMP_ORGANISMS  = [
      "Bacillus subtilis",
]

# "filtered" = only phase survivors; "all" = every input sequence
LLAMP_ON         = "filtered"

# Skip organisms that already have saved LLAMP checkpoint CSVs
LLAMP_RESUME     = True

BG, PANEL, GRID = "#1a1a1a", "#242424", "#333333"
TEXT, MUTED     = "#d4d2c8", "#888780"
PASS_C, FAIL_C  = "#1D9E75", "#E24B4A"
PHASE_COLORS    = ["#4C9AFF", "#1D9E75", "#E8A838", "#C77DFF"]

PHASE_VIEWS = [
    ("phase1",         "Phase 1",         [1, 0, 0, 0]),
    ("phase2",         "Phase 2",         [0, 1, 0, 0]),
    ("phase3",         "Phase 3",         [0, 0, 1, 0]),
    ("phase4",         "Phase 4",         [0, 0, 0, 1]),
    ("phase1-2",       "Phase 1+2",       [1, 1, 0, 0]),
    ("phase1-2-3",     "Phase 1+2+3",     [1, 1, 1, 0]),
    ("phase1-2-3-4",   "Phase 1+2+3+4",   [1, 1, 1, 1]),
]

TRAINED_ORGANISMS = [
    "Acinetobacter baumannii", "Enterococcus faecalis", "Escherichia coli",
    "Klebsiella pneumoniae", "Pseudomonas aeruginosa", "Staphylococcus aureus",
    "Staphylococcus epidermidis", "Bacillus subtilis", "Salmonella enterica",
]
ORGANISM_ALIASES = {
    "baumannii": "Acinetobacter baumannii", "a_baumannii": "Acinetobacter baumannii",
    "faecalis": "Enterococcus faecalis", "e_faecalis": "Enterococcus faecalis",
    "e_coli": "Escherichia coli", "ecoli": "Escherichia coli",
    "pneumoniae": "Klebsiella pneumoniae", "k_pneumoniae": "Klebsiella pneumoniae",
    "aeruginosa": "Pseudomonas aeruginosa", "p_aeruginosa": "Pseudomonas aeruginosa",
    "aureus": "Staphylococcus aureus", "s_aureus": "Staphylococcus aureus",
    "epidermidis": "Staphylococcus epidermidis", "s_epidermidis": "Staphylococcus epidermidis",
    "subtilis": "Bacillus subtilis", "b_subtilis": "Bacillus subtilis",
    "enterica": "Salmonella enterica", "s_enterica": "Salmonella enterica",
}

def resolve_organism(name):
    key = str(name).strip().lower()
    for canonical in TRAINED_ORGANISMS:
        if canonical.lower() == key:
            return canonical
    alias_key = key.replace(" ", "_").replace(".", "")
    if alias_key in ORGANISM_ALIASES:
        return ORGANISM_ALIASES[alias_key]
    if key in ORGANISM_ALIASES:
        return ORGANISM_ALIASES[key]
    raise ValueError(
        f"Could not resolve '{name}' to a trained organism. "
        f"Use a full name or alias from ORGANISM_ALIASES."
    )

os.makedirs(OUTPUT_DIR, exist_ok=True)

def _style_ax(ax, title="", xlabel="", ylabel=""):
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.set_title(title, color=TEXT, fontsize=12, pad=10)
    ax.set_xlabel(xlabel, color=MUTED, fontsize=10)
    ax.set_ylabel(ylabel, color=MUTED, fontsize=10)
    ax.grid(True, color=GRID, linewidth=0.4, linestyle="--", alpha=0.5)

def _fig_to_pil(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", pad_inches=0.12,
                facecolor=fig.get_facecolor(), dpi=150)
    buf.seek(0)
    return PILImage.open(buf).copy()

def load_input_df(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    if "sequence" not in df.columns:
        lower = {c.lower(): c for c in df.columns}
        if "sequence" in lower:
            df = df.rename(columns={lower["sequence"]: "sequence"})
        else:
            raise ValueError(f"CSV must contain 'sequence'. Found: {list(df.columns)}")
    df = df.copy()
    df["sequence"] = df["sequence"].astype(str).str.upper().str.strip()
    df = df[df["sequence"].str.len() > 0].reset_index(drop=True)
    return df

def run_phase2(df):
    return Phase2_NoveltyScorer(df, db_path=AMP_DB_PATH).evaluate()

def run_phase3_on_mask(df, mask):
    work = df.copy()
    orig_p1 = work["phase1_pass"].copy() if "phase1_pass" in work.columns else None
    work["phase1_pass"] = mask.fillna(False).astype(bool).values
    if "phase3_score" not in work.columns:
        work["phase3_score"] = np.nan
    try:
        work = Phase3_AMPlifyScorer(work, amplify_path=AMPLIFY_PATH).evaluate()
    except TypeError:
        work = Phase3_AMPlifyScorer(work).evaluate()
    if orig_p1 is not None:
        work["phase1_pass"] = orig_p1
    return work

def run_phase4_on_mask(df, mask):
    work = df.copy()
    orig_p1 = work["phase1_pass"].copy() if "phase1_pass" in work.columns else None
    work["phase1_pass"] = mask.fillna(False).astype(bool).values
    if "phase4_score" not in work.columns:
        work["phase4_score"] = np.nan
    work = Phase4_ToxicityScorer(work).evaluate()
    if orig_p1 is not None:
        work["phase1_pass"] = orig_p1
    return work

def run_all_phases_once(df):
    print("\n── Running full phase scoring (single pass per phase) ──")
    work = df.copy()

    work = Phase1_PhysicochemicalGate(work).evaluate()

    w2 = work.copy()
    w2["phase1_pass"] = True
    w2 = run_phase2(w2)
    for col in ["phase2_score", "phase2_blast_identity", "phase2_entropy"]:
        if col in w2.columns:
            work[col] = w2[col]

    all_mask = pd.Series(True, index=work.index)
    work = run_phase3_on_mask(work, all_mask)
    work = run_phase4_on_mask(work, all_mask)

    work["phase2_pass"] = work["phase2_score"] > PHASE2_THRESHOLD
    work["phase3_pass"] = work["phase3_score"] >= PHASE3_THRESHOLD
    work["phase4_pass"] = work["phase4_score"] >= PHASE4_THRESHOLD
    return work

def infer_phase1_fail_reason(row):
    if row.get("phase1_pass"):
        return "Passed"
    seq = row.get("sequence", "")
    if len(seq) < 6:
        return "Too Short (< 6 aa)"
    if len(seq) > 100:
        return "Too Long (> 100 aa)"
    hydro = row.get("phase1_hydrophobic_frac", 0)
    if hydro < 0.1:
        return "Too Hydrophilic (hydro < 10%)"
    if hydro > 0.73:
        return "Too Hydrophobic (hydro > 73%)"
    if row.get("phase1_charge", 0) < -2:
        return "Low Charge (< -2 at pH 7.4)"
    return "Other"

def compute_view_masks(df):
    p1 = df["phase1_pass"].fillna(False)
    p2 = df["phase2_pass"].fillna(False)
    p3 = df["phase3_pass"].fillna(False)
    p4 = df["phase4_pass"].fillna(False)
    return {
        "phase1":       p1,
        "phase2":       p2,
        "phase3":       p3,
        "phase4":       p4,
        "phase1-2":     p1 & p2,
        "phase1-2-3":   p1 & p2 & p3,
        "phase1-2-3-4": p1 & p2 & p3 & p4,
    }

def build_combination_summary(df, total):
    masks = compute_view_masks(df)
    rows = []
    for key, label, active in PHASE_VIEWS:
        m = masks[key]
        n = int(m.sum())
        rows.append({
            "view_key": key,
            "view_label": label,
            "phase1_on": active[0], "phase2_on": active[1],
            "phase3_on": active[2], "phase4_on": active[3],
            "n_pass": n,
            "pct_of_input": round(100 * n / total, 2) if total else 0,
            "pct_filtered_out": round(100 * (1 - n / total), 2) if total else 0,
        })
    return pd.DataFrame(rows), masks

def _explorer_score_series(df, col):
    if col not in df.columns:
        return [None] * len(df)
    return [None if pd.isna(v) else float(v) for v in df[col]]

def build_threshold_sweep_table(df, total):
    p1 = df["phase1_pass"].fillna(False).values
    rows = []
    for phase_key, col, op in [
        ("phase2", "phase2_score", ">"),
        ("phase3", "phase3_score", ">="),
        ("phase4", "phase4_score", ">="),
    ]:
        if col not in df.columns:
            continue
        scores = df[col].values
        valid = ~pd.isna(scores)
        chosen_thr = int({"phase2": PHASE2_THRESHOLD, "phase3": PHASE3_THRESHOLD, "phase4": PHASE4_THRESHOLD}[phase_key])
        for thr in range(0, 101):
            if op == ">":
                mask = valid & (scores > thr)
            else:
                mask = valid & (scores >= thr)
            n_standalone = int(mask.sum())
            n_cumulative = int((p1 & mask).sum())
            rows.append({
                "phase": phase_key,
                "threshold": thr,
                "n_pass_standalone": n_standalone,
                "pct_pass_standalone": round(100 * n_standalone / total, 2) if total else 0,
                "n_pass_after_p1": n_cumulative,
                "pct_pass_after_p1": round(100 * n_cumulative / total, 2) if total else 0,
                "n_dropped_standalone": total - n_standalone,
                "is_chosen_threshold": thr == chosen_thr,
            })
    return pd.DataFrame(rows)

def count_at_thresholds(df, t2, t3, t4, use_p1=True, use_p2=True, use_p3=True, use_p4=True):
    total = len(df)
    p1 = df["phase1_pass"].fillna(False).values if use_p1 else np.ones(total, dtype=bool)
    p2s = df["phase2_score"].values if "phase2_score" in df.columns else np.full(total, np.nan)
    p3s = df["phase3_score"].values if "phase3_score" in df.columns else np.full(total, np.nan)
    p4s = df["phase4_score"].values if "phase4_score" in df.columns else np.full(total, np.nan)

    m = np.ones(total, dtype=bool)
    drops = {}
    if use_p1:
        after = m & p1
        drops["phase1"] = int(m.sum() - after.sum())
        m = after
    if use_p2:
        p2ok = ~np.isnan(p2s) & (p2s > t2)
        after = m & p2ok
        drops["phase2"] = int(m.sum() - after.sum())
        m = after
    if use_p3:
        p3ok = ~np.isnan(p3s) & (p3s >= t3)
        after = m & p3ok
        drops["phase3"] = int(m.sum() - after.sum())
        m = after
    if use_p4:
        p4ok = ~np.isnan(p4s) & (p4s >= t4)
        after = m & p4ok
        drops["phase4"] = int(m.sum() - after.sum())
        m = after
    return int(m.sum()), total - int(m.sum()), drops

_PHASES_DASHBOARD_HTML = None
_LLAMP_DASHBOARD_HTML = None

def log_interactive_dashboards(run):
    rows = []
    if _PHASES_DASHBOARD_HTML:
        run.log({"dashboards/phases_interactive": wandb.Html(_PHASES_DASHBOARD_HTML, inject=False)})
        rows.append(["phases_interactive", "Phases 1–4: toggle phases, drag thresholds, see pass counts live"])
    if _LLAMP_DASHBOARD_HTML:
        run.log({"dashboards/llamp_interactive": wandb.Html(_LLAMP_DASHBOARD_HTML, inject=False)})
        rows.append(["llamp_interactive", "LLAMP: toggle organisms, recompute lowest-MIC winners live"])
    if rows:
        run.log({"dashboards/index": wandb.Table(columns=["key", "what_it_does"], data=rows)})

def _log_summary(run, metrics):
    run.summary.update({k: v for k, v in metrics.items() if v is not None})

def build_phase_retention_table(total, masks):
    stages = [
        ("Raw input", total),
        ("After Phase 1", int(masks["phase1"].sum())),
        ("After Phase 1+2", int(masks["phase1-2"].sum())),
        ("After Phase 1+2+3", int(masks["phase1-2-3"].sum())),
        ("After Phase 1+2+3+4", int(masks["phase1-2-3-4"].sum())),
    ]
    rows = []
    prev = total
    for label, n in stages:
        pct_remain = round(100 * n / total, 2) if total else 0
        pct_lost_step = round(100 * (prev - n) / prev, 2) if prev else 0
        pct_lost_total = round(100 * (total - n) / total, 2) if total else 0
        rows.append({
            "stage": label,
            "n_still_passing": n,
            "pct_still_passing": pct_remain,
            "pct_removed_this_step": pct_lost_step,
            "pct_removed_from_start": pct_lost_total,
            "n_removed_this_step": prev - n,
        })
        prev = n
    return pd.DataFrame(rows)

def compute_llamp_win_stats(wide, organisms, home_org):
    home_org = resolve_organism(home_org)
    mic_cols = [f"mic_{o}" for o in organisms]
    mic_matrix = wide[mic_cols].to_numpy(dtype=float)
    n = len(wide)
    has_valid = ~np.isnan(mic_matrix).all(axis=1)
    row_min = np.nanmin(mic_matrix, axis=1)
    is_winner = np.isclose(mic_matrix, row_min[:, None], atol=1e-9, equal_nan=False)
    is_winner[np.isnan(mic_matrix)] = False
    is_winner[~has_valid, :] = False

    home_idx = organisms.index(home_org) if home_org in organisms else -1
    home_wins = int(is_winner[:, home_idx].sum()) if home_idx >= 0 else 0
    evaluable = int(has_valid.sum())
    other_wins = {}
    for j, org in enumerate(organisms):
        if org == home_org:
            continue
        cnt = int(is_winner[:, j].sum())
        if cnt:
            other_wins[org] = cnt

    stats = []
    for j, org in enumerate(organisms):
        wins = int(is_winner[:, j].sum())
        stats.append({
            "organism": org,
            "role": "HOME" if org == home_org else "competitor",
            "win_count": wins,
            "win_rate_pct": round(100 * wins / n, 2) if n else 0,
            "win_rate_of_evaluable_pct": round(100 * wins / evaluable, 2) if evaluable else 0,
        })
    return {
        "home_wins": home_wins,
        "other_wins": other_wins,
        "ties_or_na": int((~has_valid).sum()),
        "evaluable": evaluable,
        "stats_df": pd.DataFrame(stats),
        "is_winner": is_winner,
        "has_valid": has_valid,
    }

def build_llamp_explorer_html(wide, organisms, home_org):
    home_org = resolve_organism(home_org)
    mic_cols = [f"mic_{o}" for o in organisms]
    mics = []
    for _, row in wide.iterrows():
        mics.append([None if pd.isna(row[c]) else float(row[c]) for c in mic_cols])

    payload = {
        "total": len(wide),
        "homeOrg": home_org,
        "organisms": organisms,
        "shortNames": {o: (f"{o.split()[0][0]}. {o.split()[-1]}" if " " in o else o) for o in organisms},
        "mics": mics,
    }
    data_json = json.dumps(payload)
    toggles = ""
    for i, org in enumerate(organisms):
        home_mark = " ★ HOME" if org == home_org else ""
        checked = "checked"
        toggles += (
            f'<label class="toggle active" id="tl{i}">'
            f'<input type="checkbox" id="o{i}" data-idx="{i}" {checked}> '
            f'{org}{home_mark}</label>\n'
        )

    return f"""<!DOCTYPE html>
<html><head><meta charset="utf-8">
<style>
  :root {{ --bg:#1a1a1a; --panel:#242424; --grid:#333; --text:#d4d2c8; --muted:#888780;
           --pass:#1D9E75; --fail:#E24B4A; --home:#E8A838; --accent:#4C9AFF; }}
  body {{ font-family:'Segoe UI',system-ui,sans-serif; background:var(--bg); color:var(--text);
          padding:24px; max-width:1200px; margin:0 auto; }}
  h1 {{ font-size:1.4rem; margin:0 0 4px; }}
  h2 {{ font-size:.82rem; color:var(--muted); text-transform:uppercase; letter-spacing:.08em;
        margin:22px 0 10px; border-bottom:1px solid var(--grid); padding-bottom:5px; }}
  .sub {{ color:var(--muted); margin-bottom:14px; }}
  .toggles {{ display:flex; flex-wrap:wrap; gap:8px; }}
  .toggle {{ display:flex; align-items:center; gap:6px; background:var(--panel); padding:7px 11px;
             border-radius:8px; border:1px solid var(--grid); cursor:pointer; font-size:.78rem; max-width:100%; }}
  .toggle.active {{ border-color:var(--pass); }}
  .toggle.home {{ border-color:var(--home); }}
  .toggle input {{ accent-color:var(--pass); flex-shrink:0; }}
  .metrics {{ display:grid; grid-template-columns:repeat(4,1fr); gap:12px; margin:16px 0; }}
  @media(max-width:800px){{ .metrics {{ grid-template-columns:repeat(2,1fr); }} }}
  .metric {{ background:var(--panel); border:1px solid var(--grid); border-radius:10px; padding:14px; text-align:center; }}
  .metric .val {{ font-size:1.6rem; font-weight:700; color:var(--pass); }}
  .metric .lbl {{ font-size:.72rem; color:var(--muted); margin-top:4px; }}
  .metric.home .val {{ color:var(--home); }}
  table {{ width:100%; border-collapse:collapse; margin-top:8px; }}
  th,td {{ padding:8px 10px; border-bottom:1px solid var(--grid); font-size:.82rem; text-align:left; }}
  th {{ color:var(--muted); font-size:.7rem; text-transform:uppercase; }}
  tr.home-row {{ background:#2a2418; }}
  .bar-cell {{ min-width:120px; }}
  .bar {{ height:8px; background:var(--accent); border-radius:4px; }}
  .bar.home {{ background:var(--home); }}
  .delta {{ background:#1a2a3a; border:1px solid var(--accent); border-radius:8px; padding:12px; font-size:.88rem; margin:12px 0; }}
</style></head><body>
<h1>LLAMP Organism Competition Explorer</h1>
<p class="sub">{len(wide):,} peptides - target: <strong style="color:var(--home)">{home_org}</strong> - toggle organisms to see new winners</p>
<h2>Active organisms in competition</h2>
<div class="toggles">{toggles}</div>
<div class="metrics">
  <div class="metric"><div class="val" id="m-eval">—</div><div class="lbl">Evaluable peptides</div></div>
  <div class="metric home"><div class="val" id="m-home">—</div><div class="lbl">Home win rate</div></div>
  <div class="metric"><div class="val" id="m-leader">—</div><div class="lbl">Current top winner</div></div>
  <div class="metric"><div class="val" id="m-na">—</div><div class="lbl">Invalid MIC count</div></div>
</div>
<div class="delta" id="delta">Select organisms above — winners recomputed among active set only.</div>
<h2>Win rates (lowest MIC wins)</h2>
<table><thead><tr>
  <th>Organism</th><th>Role</th><th>Wins</th><th>% of all</th><th>% of evaluable</th><th>Share</th>
</tr></thead><tbody id="win-rows"></tbody></table>
<script>
const D={data_json};
function activeIdx() {{
  return D.organisms.map((_,i)=>document.getElementById('o'+i).checked?i:null).filter(i=>i!==null);
}}
function compute() {{
  const active=activeIdx();
  const n=D.total;
  if(active.length===0) return {{evaluable:0,na:n,stats:[],homePct:0,leader:'—'}};
  const wins=active.map(()=>0);
  let evaluable=0, na=0;
  for(let s=0;s<n;s++) {{
    let best=Infinity, bestList=[], any=false;
    for(const j of active) {{
      const v=D.mics[s][j];
      if(v!==null && !isNaN(v)) {{ any=true; if(v<best-1e-9){{best=v;bestList=[j];}} else if(Math.abs(v-best)<1e-9) bestList.push(j); }}
    }}
    if(!any) {{ na++; continue; }}
    evaluable++;
    for(const j of bestList) wins[active.indexOf(j)]++;
  }}
  const stats=active.map((orgIdx,i)=>({{
    org:D.organisms[orgIdx], idx:orgIdx,
    wins:wins[i],
    pctAll:(100*wins[i]/n).toFixed(1),
    pctEval:evaluable?(100*wins[i]/evaluable).toFixed(1):'0.0',
    isHome:D.organisms[orgIdx]===D.homeOrg
  }})).sort((a,b)=>b.wins-a.wins);
  const homeStat=stats.find(s=>s.isHome);
  const leader=stats.length?stats[0].org:'—';
  return {{evaluable,na,stats,homePct:homeStat?homeStat.pctAll:'0.0',leader}};
}}
function upd() {{
  const active=activeIdx();
  D.organisms.forEach((_,i)=>{{
    const el=document.getElementById('tl'+i);
    const on=document.getElementById('o'+i).checked;
    el.classList.toggle('active',on);
    el.classList.toggle('home',D.organisms[i]===D.homeOrg);
  }});
  const r=compute();
  document.getElementById('m-eval').textContent=r.evaluable.toLocaleString()+' ('+(100*r.evaluable/D.total).toFixed(1)+'%)';
  document.getElementById('m-home').textContent=r.homePct+'%';
  document.getElementById('m-leader').textContent=D.shortNames[r.leader]||r.leader;
  document.getElementById('m-na').textContent=r.na.toLocaleString()+' ('+(100*r.na/D.total).toFixed(1)+'%)';
  const maxW=Math.max(...r.stats.map(s=>s.wins),1);
  let rows='';
  for(const s of r.stats) {{
    const barW=Math.round(100*s.wins/maxW);
    rows+=`<tr class="${{s.isHome?'home-row':''}}"><td>${{s.org}}</td><td>${{s.isHome?'HOME':'competitor'}}</td>
      <td>${{s.wins.toLocaleString()}}</td><td>${{s.pctAll}}%</td><td>${{s.pctEval}}%</td>
      <td class="bar-cell"><div class="bar ${{s.isHome?'home':''}}" style="width:${{barW}}%"></div></td></tr>`;
  }}
  document.getElementById('win-rows').innerHTML=rows||'<tr><td colspan="6">No organisms selected</td></tr>';
  if(active.length<2) document.getElementById('delta').textContent='Select at least 2 organisms to compare winners.';
  else {{
    const homeIn=active.some(i=>D.organisms[i]===D.homeOrg)?'':' (home excluded)';
    document.getElementById('delta').textContent=
      `Among ${{active.length}} active organisms${{homeIn}}: ${{r.leader}} wins most often (${{r.stats[0]?.pctAll||0}}% of all peptides). Home wins ${{r.homePct}}%.`;
  }}
}}
D.organisms.forEach((_,i)=>document.getElementById('o'+i).addEventListener('change',upd));
upd();
</script></body></html>"""

def build_pipeline_explorer_html(df, combo_df, total, chosen):
    payload = {
        "total": total,
        "p1": df["phase1_pass"].fillna(False).astype(bool).tolist(),
        "p2": _explorer_score_series(df, "phase2_score"),
        "p3": _explorer_score_series(df, "phase3_score"),
        "p4": _explorer_score_series(df, "phase4_score"),
        "chosen": {"t2": float(chosen["p2"]), "t3": float(chosen["p3"]), "t4": float(chosen["p4"])},
    }
    data_json = json.dumps(payload)
    rows_html = ""
    for _, r in combo_df.iterrows():
        rows_html += (
            f"<tr data-p1='{int(r['phase1_on'])}' data-p2='{int(r['phase2_on'])}' "
            f"data-p3='{int(r['phase3_on'])}' data-p4='{int(r['phase4_on'])}' "
            f"data-label='{r['view_label']}'><td>{r['view_label']}</td>"
            f"<td>{int(r['n_pass']):,}</td><td>{r['pct_of_input']:.1f}%</td></tr>"
        )
    c = chosen
    return f"""<!DOCTYPE html>
<html><head><meta charset="utf-8">
<style>
  :root {{ --bg:#1a1a1a; --panel:#242424; --grid:#333; --text:#d4d2c8; --muted:#888780;
           --pass:#1D9E75; --fail:#E24B4A; --accent:#4C9AFF; --chosen:#E8A838; }}
  body {{ font-family:'Segoe UI',system-ui,sans-serif; background:var(--bg); color:var(--text);
          padding:24px; max-width:1100px; margin:0 auto; }}
  h1 {{ font-size:1.5rem; margin:0 0 4px; }}
  h2 {{ font-size:.85rem; color:var(--muted); text-transform:uppercase; letter-spacing:.08em;
        margin:24px 0 10px; border-bottom:1px solid var(--grid); padding-bottom:6px; }}
  .sub {{ color:var(--muted); margin-bottom:16px; }}
  .toggles {{ display:flex; gap:10px; flex-wrap:wrap; }}
  .toggle {{ display:flex; align-items:center; gap:8px; background:var(--panel); padding:8px 12px;
             border-radius:8px; border:1px solid var(--grid); cursor:pointer; font-size:.9rem; }}
  .toggle.active {{ border-color:var(--pass); }}
  .toggle input {{ accent-color:var(--pass); }}
  .sliders {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(260px,1fr)); gap:14px; margin-top:8px; }}
  .slider-card {{ background:var(--panel); border:1px solid var(--grid); border-radius:10px; padding:14px; }}
  .slider-card.off {{ opacity:.4; pointer-events:none; }}
  .slider-card .head {{ display:flex; justify-content:space-between; }}
  .slider-card .val {{ font-size:1.4rem; font-weight:700; color:var(--accent); }}
  .chosen-mark {{ font-size:.72rem; color:var(--chosen); margin:4px 0 6px; }}
  input[type=range] {{ width:100%; accent-color:var(--accent); }}
  .compare {{ display:grid; grid-template-columns:1fr 1fr; gap:14px; margin:16px 0; }}
  @media(max-width:640px){{ .compare {{ grid-template-columns:1fr; }} }}
  .box {{ background:var(--panel); border-radius:10px; padding:16px; border:1px solid var(--grid); }}
  .box.a {{ border-color:var(--chosen); }}
  .box.b {{ border-color:var(--pass); }}
  .box h3 {{ margin:0 0 10px; font-size:.9rem; }}
  .stat {{ display:flex; justify-content:space-between; padding:5px 0; border-bottom:1px solid var(--grid); font-size:.88rem; }}
  .stat:last-child {{ border:none; }}
  .pass {{ color:var(--pass); font-weight:700; }}
  .fail {{ color:var(--fail); font-weight:700; }}
  .delta {{ background:#2a2418; border:1px solid var(--chosen); border-radius:8px; padding:12px 16px; font-size:.9rem; }}
  .drops {{ display:grid; grid-template-columns:repeat(4,1fr); gap:8px; }}
  .drop-chip {{ background:var(--panel); border:1px solid var(--grid); border-radius:8px; padding:10px; text-align:center; }}
  .drop-chip .num {{ font-size:1.1rem; font-weight:700; color:var(--fail); }}
  .drop-chip .lbl {{ font-size:.68rem; color:var(--muted); }}
  .metric-box {{ display:grid; grid-template-columns:repeat(5,1fr); gap:10px; margin:12px 0; }}
  @media(max-width:900px){{ .metric-box {{ grid-template-columns:repeat(2,1fr); }} }}
  .metric-box .metric {{ background:var(--panel); border:1px solid var(--grid); border-radius:10px; padding:12px; text-align:center; }}
  .metric-box .metric .val {{ font-size:1.15rem; font-weight:700; color:var(--pass); }}
  .metric-box .metric .lbl {{ font-size:.65rem; color:var(--muted); margin-top:4px; line-height:1.3; }}
  table {{ width:100%; border-collapse:collapse; }}
  th,td {{ padding:7px 10px; border-bottom:1px solid var(--grid); font-size:.82rem; }}
  th {{ color:var(--muted); text-transform:uppercase; font-size:.7rem; }}
  tr.on {{ background:#1a3d2e; }}
</style></head><body>
<h1>Pipeline & Threshold Explorer</h1>
<p class="sub">{total:,} peptides · toggle phases · drag thresholds · compare vs your chosen run</p>
<h2>Active phases</h2>
<div class="toggles">
  <label class="toggle active" id="tl1"><input type="checkbox" id="fp1" checked> P1 Physicochemical</label>
  <label class="toggle active" id="tl2"><input type="checkbox" id="fp2" checked> P2 Novelty</label>
  <label class="toggle active" id="tl3"><input type="checkbox" id="fp3" checked> P3 AMPlify</label>
  <label class="toggle active" id="tl4"><input type="checkbox" id="fp4" checked> P4 Safety</label>
</div>
<h2>Threshold draggers</h2>
<div class="sliders">
  <div class="slider-card" id="sc2"><div class="head"><span>P2 Novelty</span><span class="val" id="tv2">{int(c['p2'])}</span></div>
    <div class="chosen-mark">Your run: &gt; {c['p2']}</div><input type="range" id="t2" min="0" max="100" value="{int(c['p2'])}"></div>
  <div class="slider-card" id="sc3"><div class="head"><span>P3 AMPlify</span><span class="val" id="tv3">{int(c['p3'])}</span></div>
    <div class="chosen-mark">Your run: ≥ {c['p3']}</div><input type="range" id="t3" min="0" max="100" value="{int(c['p3'])}"></div>
  <div class="slider-card" id="sc4"><div class="head"><span>P4 Safety</span><span class="val" id="tv4">{int(c['p4'])}</span></div>
    <div class="chosen-mark">Your run: ≥ {c['p4']}</div><input type="range" id="t4" min="0" max="100" value="{int(c['p4'])}"></div>
</div>
<h2>Your settings vs exploring</h2>
<div class="compare">
  <div class="box a"><h3>★ Your chosen thresholds</h3>
    <div class="stat"><span>Passing</span><span class="pass" id="cp">—</span></div>
    <div class="stat"><span>Dropped</span><span class="fail" id="cd">—</span></div>
    <div class="stat"><span>Retention</span><span id="cx">—</span></div></div>
  <div class="box b"><h3>◉ Exploring (sliders)</h3>
    <div class="stat"><span>Passing</span><span class="pass" id="ep">—</span></div>
    <div class="stat"><span>Dropped</span><span class="fail" id="ed">—</span></div>
    <div class="stat"><span>Retention</span><span id="ex">—</span></div></div>
</div>
<div class="delta" id="delta">Move sliders to compare.</div>
<h2>Cumulative retention (chosen thresholds, all phases on)</h2>
<div class="metric-box" id="retention-ladder"></div>
<h2>Per-phase drops (exploring)</h2>
<div class="drops">
  <div class="drop-chip"><div class="num" id="d1">0</div><div class="lbl">P1 dropped</div></div>
  <div class="drop-chip"><div class="num" id="d2">0</div><div class="lbl">P2 dropped</div></div>
  <div class="drop-chip"><div class="num" id="d3">0</div><div class="lbl">P3 dropped</div></div>
  <div class="drop-chip"><div class="num" id="d4">0</div><div class="lbl">P4 dropped</div></div>
</div>
<h2>Preset views</h2>
<table><thead><tr><th>View</th><th>Count</th><th>%</th></tr></thead><tbody id="presets">{rows_html}</tbody></table>
<script>
const D={data_json};
const presets=document.querySelectorAll('#presets tr');
function phases(){{return[fp1.checked,fp2.checked,fp3.checked,fp4.checked];}}
function thr(){{return{{t2:+t2.value,t3:+t3.value,t4:+t4.value}};}}
function run(useP1,useP2,useP3,useP4,t2,t3,t4){{
  let m=Array(D.total).fill(true), drops={{p1:0,p2:0,p3:0,p4:0}}, b;
  if(useP1){{b=m.filter(Boolean).length;for(let i=0;i<D.total;i++)if(m[i]&&!D.p1[i])m[i]=false;drops.p1=b-m.filter(Boolean).length;}}
  if(useP2){{b=m.filter(Boolean).length;for(let i=0;i<D.total;i++)if(m[i]&&(D.p2[i]===null||D.p2[i]<=t2))m[i]=false;drops.p2=b-m.filter(Boolean).length;}}
  if(useP3){{b=m.filter(Boolean).length;for(let i=0;i<D.total;i++)if(m[i]&&(D.p3[i]===null||D.p3[i]<t3))m[i]=false;drops.p3=b-m.filter(Boolean).length;}}
  if(useP4){{b=m.filter(Boolean).length;for(let i=0;i<D.total;i++)if(m[i]&&(D.p4[i]===null||D.p4[i]<t4))m[i]=false;drops.p4=b-m.filter(Boolean).length;}}
  const p=m.filter(Boolean).length;return{{p,d:D.total-p,drops}};
}}
function pct(n){{return(100*n/D.total).toFixed(1)+'%';}}
function upd(){{
  const ph=phases(), t=thr();
  ['tl1','tl2','tl3','tl4'].forEach((id,i)=>document.getElementById(id).classList.toggle('active',ph[i]));
  ['sc2','sc3','sc4'].forEach((id,i)=>document.getElementById(id).classList.toggle('off',!ph[i+1]));
  tv2.textContent=t.t2; tv3.textContent=t.t3; tv4.textContent=t.t4;
  const ch=run(true,true,true,true,D.chosen.t2,D.chosen.t3,D.chosen.t4);
  cp.textContent=ch.p.toLocaleString(); cd.textContent=ch.d.toLocaleString(); cx.textContent=pct(ch.p);
  const ladder=document.getElementById('retention-ladder');
  if(ladder){{
    const stages=['Raw','P1','P1+P2','P1+P2+P3','P1+P2+P3+P4'];
    const counts=[D.total];
    const r=run(true,true,true,true,D.chosen.t2,D.chosen.t3,D.chosen.t4);
    let c=D.total; ladder.innerHTML='';
    for(let si=1;si<=4;si++){{
      const sub=run(true,si>=2,si>=3,si>=4,D.chosen.t2,D.chosen.t3,D.chosen.t4);
      counts.push(sub.p);
    }}
    stages.forEach((s,i)=>{{
      const n=counts[i], p=(100*n/D.total).toFixed(1);
      const drop=i>0?((100*(counts[i-1]-n)/counts[i-1]).toFixed(1)):0;
      ladder.innerHTML+=`<div class="metric"><div class="val">${{p}}%</div><div class="lbl">${{s}}<br>${{n.toLocaleString()}} left${{i>0?' · −'+drop+'% step':''}}</div></div>`;
    }});
  }}
  const ex=run(ph[0],ph[1],ph[2],ph[3],t.t2,t.t3,t.t4);
  ep.textContent=ex.p.toLocaleString(); ed.textContent=ex.d.toLocaleString(); ex.textContent=pct(ex.p);
  const fmtDrop=(n)=>`${{n.toLocaleString()}} (${{(100*n/D.total).toFixed(1)}}%)`;
  d1.textContent=fmtDrop(ex.drops.p1); d2.textContent=fmtDrop(ex.drops.p2);
  d3.textContent=fmtDrop(ex.drops.p3); d4.textContent=fmtDrop(ex.drops.p4);
  presets.forEach(r=>{{const rp=[r.dataset.p1==='1',r.dataset.p2==='1',r.dataset.p3==='1',r.dataset.p4==='1'];
    r.classList.toggle('on',rp.every((v,i)=>v===ph[i]));}});
  const diff=ex.p-ch.p;
  if(diff===0&&t.t2===D.chosen.t2&&t.t3===D.chosen.t3&&t.t4===D.chosen.t4&&ph.every(Boolean))
    delta.innerHTML='✓ Matches your chosen run settings exactly.';
  else if(diff>0) delta.innerHTML=`<span class="pass">+${{diff.toLocaleString()}}</span> more peptides pass vs your chosen thresholds (${{pct(ex.p)}} vs ${{pct(ch.p)}}).`;
  else if(diff<0) delta.innerHTML=`<span class="fail">${{(-diff).toLocaleString()}}</span> fewer peptides pass — stricter than your chosen run.`;
  else delta.innerHTML='Same total pass count, different phase/threshold mix.';
}}
['fp1','fp2','fp3','fp4','t2','t3','t4'].forEach(id=>document.getElementById(id).addEventListener('input',upd));
upd();
</script></body></html>"""

def log_threshold_sensitivity(run, df, total):
    sweep = build_threshold_sweep_table(df, total)
    if sweep.empty:
        return sweep
    run.log({"phases/tables/threshold_sweep": wandb.Table(dataframe=sweep)})
    run.log({"phases/tables/threshold_at_your_settings": wandb.Table(
        dataframe=sweep[sweep["is_chosen_threshold"]].reset_index(drop=True)
    )})

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), facecolor=BG)
    for ax, (pk, title, chosen_thr) in zip(axes, [
        ("phase2", "Phase 2 Novelty — % passing vs threshold", PHASE2_THRESHOLD),
        ("phase3", "Phase 3 AMPlify — % passing vs threshold", PHASE3_THRESHOLD),
        ("phase4", "Phase 4 Safety — % passing vs threshold", PHASE4_THRESHOLD),
    ]):
        sub = sweep[sweep["phase"] == pk]
        if sub.empty:
            continue
        ax.plot(sub["threshold"], sub["pct_pass_standalone"], color="#4C9AFF", lw=2, label="All input")
        ax.plot(sub["threshold"], sub["pct_pass_after_p1"], color="#1D9E75", lw=2, ls="--", label="After Phase 1")
        ax.axvline(chosen_thr, color="#E8A838", lw=2.5, label=f"Your threshold ({chosen_thr:g})")
        row = sub[sub["threshold"] == int(chosen_thr)]
        if len(row):
            y = float(row.iloc[0]["pct_pass_after_p1"])
            ax.scatter([chosen_thr], [y], s=70, c="#E8A838", zorder=5, edgecolors="white")
            ax.annotate(f"{y:.1f}% pass", xy=(chosen_thr, y), xytext=(8, 8),
                        textcoords="offset points", color="#E8A838", fontsize=8)
        _style_ax(ax, title, xlabel="Threshold value", ylabel="% of input passing")
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
        ax.set_ylim(0, 105)
        ax.legend(facecolor=PANEL, edgecolor=GRID, labelcolor=TEXT, fontsize=7)
    fig.suptitle(
        "Threshold pass rates — move sliders in the interactive dashboard to explore; "
        "gold dot = your current settings",
        color=TEXT, fontsize=10,
    )
    fig.tight_layout()
    run.log({"phases/plots/threshold_pass_rate_curves": wandb.Image(_fig_to_pil(fig))})
    plt.close(fig)

    return sweep

def log_phase_dashboard(run, df, combo_df, masks, total):
    thresholds = {"p2": PHASE2_THRESHOLD, "p3": PHASE3_THRESHOLD, "p4": PHASE4_THRESHOLD}

    global _PHASES_DASHBOARD_HTML
    html = build_pipeline_explorer_html(df, combo_df, total, thresholds)
    _PHASES_DASHBOARD_HTML = html
    html_path = os.path.join(OUTPUT_DIR, "phases_interactive_dashboard.html")
    with open(html_path, "w") as f:
        f.write(html)

    log_threshold_sensitivity(run, df, total)

    retention_df = build_phase_retention_table(total, masks)
    run.log({"phases/tables/peptides_still_passing_by_stage": wandb.Table(dataframe=retention_df)})

    n_chosen, n_drop_chosen, phase_drops = count_at_thresholds(
        df, PHASE2_THRESHOLD, PHASE3_THRESHOLD, PHASE4_THRESHOLD
    )
    _log_summary(run, {
        "thresholds/chosen/phase2": PHASE2_THRESHOLD,
        "thresholds/chosen/phase3": PHASE3_THRESHOLD,
        "thresholds/chosen/phase4": PHASE4_THRESHOLD,
        "thresholds/chosen/n_pass_full_pipeline": n_chosen,
        "thresholds/chosen/n_dropped_full_pipeline": n_drop_chosen,
        "thresholds/chosen/pct_still_passing_full_pipeline": round(100 * n_chosen / total, 2) if total else 0,
        "thresholds/chosen/drop_at_phase1": phase_drops.get("phase1", 0),
        "thresholds/chosen/drop_at_phase2": phase_drops.get("phase2", 0),
        "thresholds/chosen/drop_at_phase3": phase_drops.get("phase3", 0),
        "thresholds/chosen/drop_at_phase4": phase_drops.get("phase4", 0),
    })
    for _, r in combo_df.iterrows():
        _log_summary(run, {
            f"views/{r['view_key']}/n_pass": int(r["n_pass"]),
            f"views/{r['view_key']}/pct_of_input": float(r["pct_of_input"]),
            f"views/{r['view_key']}/pct_filtered_out": float(r["pct_filtered_out"]),
        })

    run.log({"phases/tables/phase_combinations": wandb.Table(dataframe=combo_df)})

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), facecolor=BG)
    stages = retention_df["stage"].tolist()
    pct_rem = retention_df["pct_still_passing"].tolist()
    pct_step = retention_df["pct_removed_this_step"].tolist()
    x = np.arange(len(stages))
    bars = axes[0].bar(x, pct_rem, color=[PASS_C if i == len(stages)-1 else "#4C9AFF" for i in range(len(stages))], edgecolor=PANEL)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(stages, rotation=20, ha="right", fontsize=8)
    _style_ax(axes[0], "Peptides still passing after each phase", ylabel="% of your input still passing")
    axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
    for bar, p, n in zip(bars, pct_rem, retention_df["n_still_passing"]):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                     f"{p:.1f}%\n({int(n):,})", ha="center", va="bottom", color=TEXT, fontsize=7)
    step_labels = stages[1:]
    bars2 = axes[1].bar(range(len(step_labels)), pct_step[1:], color=FAIL_C, edgecolor=PANEL)
    axes[1].set_xticks(range(len(step_labels)))
    axes[1].set_xticklabels([f"Drop at\n{s.replace('After ', '')}" for s in step_labels], fontsize=7)
    _style_ax(axes[1], "Peptides removed at each step", ylabel="% removed from previous step")
    axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
    for bar, p, nd in zip(bars2, pct_step[1:], retention_df["n_removed_this_step"].iloc[1:]):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f"{p:.1f}%\n({int(nd):,})", ha="center", va="bottom", color=TEXT, fontsize=7)
    fig.suptitle(f"Of {total:,} input peptides — how many still pass after each phase (your thresholds)", color=TEXT, fontsize=11)
    fig.tight_layout()
    run.log({"phases/plots/peptides_still_passing_by_stage": wandb.Image(_fig_to_pil(fig))})
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(11, 5), facecolor=BG)
    colors = [PHASE_COLORS[i] if i < 4 else "#1D9E75" for i in range(len(combo_df))]
    bars = ax.bar(combo_df["view_label"], combo_df["pct_of_input"], color=colors, edgecolor=PANEL, linewidth=0.5)
    _style_ax(ax, "How many peptides pass each phase combination?", ylabel="% of your input that pass")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    for bar, n, pct in zip(bars, combo_df["n_pass"], combo_df["pct_of_input"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{n:,}\n({pct:.1f}%)", ha="center", va="bottom", color=TEXT, fontsize=8)
    plt.xticks(rotation=25, ha="right")
    fig.tight_layout()
    run.log({"phases/plots/pass_rate_by_phase_combo": wandb.Image(_fig_to_pil(fig))})
    plt.close(fig)

    funnel_labels = ["All input", "After Phase 1", "After P1+P2", "After P1+P2+P3", "After full pipeline"]
    funnel_counts = [
        total,
        int(masks["phase1"].sum()),
        int(masks["phase1-2"].sum()),
        int(masks["phase1-2-3"].sum()),
        int(masks["phase1-2-3-4"].sum()),
    ]
    fig, ax = plt.subplots(figsize=(10, 5.5), facecolor=BG)
    pct_funnel = [100 * c / total for c in funnel_counts]
    colors_f = [MUTED] + [PHASE_COLORS[i] for i in range(4)]
    ax.barh(funnel_labels[::-1], pct_funnel[::-1], color=colors_f[::-1], edgecolor=PANEL)
    _style_ax(ax, f"Pipeline funnel — % of {total:,} input peptides still passing", xlabel="% still passing")
    ax.set_xlim(0, 105)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    rev_counts = funnel_counts[::-1]
    rev_pct = pct_funnel[::-1]
    for i, (lbl, c, pct) in enumerate(zip(funnel_labels[::-1], rev_counts, rev_pct)):
        ax.text(pct + 0.8, i, f"{pct:.1f}%  ({c:,} peptides)", va="center", color=TEXT, fontsize=9)
        if i + 1 < len(rev_counts) and rev_counts[i]:
            drop_pct = 100 * (rev_counts[i] - rev_counts[i + 1]) / rev_counts[i]
            ax.annotate(f"−{drop_pct:.1f}%", xy=(pct / 2, i + 0.45), ha="center", color=FAIL_C, fontsize=7)
    fig.tight_layout()
    run.log({"phases/plots/pipeline_funnel": wandb.Image(_fig_to_pil(fig))})
    plt.close(fig)

    if "phase1_pass" in df.columns:
        df = df.copy()
        df["phase1_fail_reason"] = df.apply(infer_phase1_fail_reason, axis=1)
        fail = df.loc[~df["phase1_pass"], "phase1_fail_reason"].value_counts()
        if len(fail):
            fig, ax = plt.subplots(figsize=(9, 5), facecolor=BG)
            fail_rev = fail.iloc[::-1]
            fail_total = int(fail.sum())
            bars_p1 = ax.barh(fail_rev.index, fail_rev.values, color=FAIL_C, edgecolor=PANEL)
            _style_ax(ax, f"Why peptides failed Phase 1 ({100*fail_total/total:.1f}% of all input)",
                      xlabel="Number of peptides")
            for bar, val in zip(bars_p1, fail_rev.values):
                pct_fail = 100 * val / fail_total if fail_total else 0
                pct_all = 100 * val / total if total else 0
                ax.text(bar.get_width() + fail_total * 0.01, bar.get_y() + bar.get_height() / 2,
                        f"{val:,} ({pct_fail:.1f}% of failures · {pct_all:.1f}% of all)",
                        va="center", color=TEXT, fontsize=7)
            fig.tight_layout()
            run.log({"phases/plots/phase1_failure_reasons": wandb.Image(_fig_to_pil(fig))})
            plt.close(fig)
            fail_tbl = fail.reset_index()
            fail_tbl.columns = ["reason", "count"]
            fail_tbl["count"] = pd.to_numeric(fail_tbl["count"], errors="coerce").fillna(0).astype(int)
            fail_tbl["pct_of_failures"] = (fail_tbl["count"] / fail_total * 100).round(1)
            fail_tbl["pct_of_all_input"] = (fail_tbl["count"] / total * 100).round(1)
            run.log({"phases/tables/phase1_failure_breakdown": wandb.Table(dataframe=fail_tbl)})

    for col, thr, name in [
        ("phase2_score", PHASE2_THRESHOLD, "Phase 2 Novelty"),
        ("phase3_score", PHASE3_THRESHOLD, "Phase 3 AMPlify"),
        ("phase4_score", PHASE4_THRESHOLD, "Phase 4 Safety"),
    ]:
        if col not in df.columns:
            continue
        scores = pd.to_numeric(df[col], errors="coerce").dropna()
        if scores.empty:
            continue
        prefix = col.replace("_score", "")
        op = ">" if col == "phase2_score" else ">="
        n_pass = int((scores > thr).sum()) if col == "phase2_score" else int((scores >= thr).sum())
        _log_summary(run, {
            f"scores/{prefix}/mean": float(scores.mean()),
            f"scores/{prefix}/median": float(scores.median()),
            f"scores/{prefix}/std": float(scores.std()) if len(scores) > 1 else 0,
            f"scores/{prefix}/pct_pass": round(100 * n_pass / len(scores), 2),
            f"scores/{prefix}/n_pass": n_pass,
        })

        fig_pf, ax_pf = plt.subplots(figsize=(5, 4), facecolor=BG)
        n_fail = len(scores) - n_pass
        ax_pf.bar(["Pass", "Fail"], [n_pass, n_fail], color=[PASS_C, FAIL_C], edgecolor=PANEL)
        _style_ax(ax_pf, f"{name} — pass vs fail at threshold {thr:g} (% of scored peptides)", ylabel="Count")
        for xi, (val, lbl) in enumerate([(n_pass, "Pass"), (n_fail, "Fail")]):
            ax_pf.text(xi, val + len(scores)*0.01, f"{val:,}\n({100*val/len(scores):.1f}%)",
                       ha="center", color=TEXT, fontsize=9, fontweight="bold")
        fig_pf.tight_layout()
        run.log({f"phases/plots/{prefix}_pass_fail_pct": wandb.Image(_fig_to_pil(fig_pf))})
        plt.close(fig_pf)
        pct_pass_all = 100 * n_pass / len(scores)
        pct_fail_all = 100 - pct_pass_all
        pct_pass_input = 100 * n_pass / total
        pct_fail_input = 100 * n_fail / total
        fig, ax = plt.subplots(figsize=(8, 4.5), facecolor=BG)
        smin, smax = float(scores.min()), float(scores.max())
        if col == "phase2_score":
            ax.axvspan(thr, smax + 1, color=PASS_C, alpha=0.12)
            ax.axvspan(smin - 1, thr, color=FAIL_C, alpha=0.12)
        else:
            ax.axvspan(thr, smax + 1, color=PASS_C, alpha=0.12)
            ax.axvspan(smin - 1, thr, color=FAIL_C, alpha=0.12)
        ax.hist(scores, bins=40, color="#4C9AFF", alpha=0.85, edgecolor=PANEL)
        ax.axvline(thr, color=FAIL_C, linestyle="--", linewidth=2, label=f"threshold={thr:g}")
        ax.axvline(scores.mean(), color="#E8A838", linestyle=":", linewidth=2, label=f"mean={scores.mean():.1f}")
        _style_ax(ax, f"{name} — score distribution",
                  xlabel="Score", ylabel="Count")
        ax.text(0.02, 0.97, f"PASS: {n_pass:,} ({pct_pass_all:.1f}% scored · {pct_pass_input:.1f}% of all input)",
                transform=ax.transAxes, ha="left", va="top", color=PASS_C, fontsize=8, fontweight="bold")
        ax.text(0.02, 0.84, f"FAIL: {n_fail:,} ({pct_fail_all:.1f}% scored · {pct_fail_input:.1f}% of all input)",
                transform=ax.transAxes, ha="left", va="top", color=FAIL_C, fontsize=8, fontweight="bold")
        ax.legend(facecolor=PANEL, edgecolor=GRID, labelcolor=TEXT, fontsize=8, loc="upper right")
        fig.tight_layout()
        run.log({f"phases/plots/{prefix}_score_distribution": wandb.Image(_fig_to_pil(fig))})
        plt.close(fig)

    passed = df[df["phase1_pass"]] if "phase1_pass" in df.columns else df
    feat_specs = [
        ("phase1_length", "Length (aa)"),
        ("phase1_charge", "Net Charge @ pH 7.4"),
        ("phase1_hydrophobic_frac", "Hydrophobic Fraction"),
        ("phase1_hydrophobic_moment", "Hydrophobic Moment"),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), facecolor=BG)
    for ax, (col, label) in zip(axes.flat, feat_specs):
        if col not in passed.columns:
            continue
        vals = passed[col].dropna()
        ax.set_facecolor(PANEL)
        ax.hist(vals, bins=30, color=PASS_C, alpha=0.8, edgecolor=PANEL)
        pct_of_input = 100 * len(vals) / total if total else 0
        med = float(vals.median()) if len(vals) else 0
        ax.axvline(med, color="#E8A838", ls=":", lw=1.5)
        _style_ax(ax, f"{label} (median={med:.2g})", xlabel=label, ylabel="Count")
        ax.text(0.98, 0.95, f"n={len(vals):,}\n({pct_of_input:.1f}% of input)",
                transform=ax.transAxes, ha="right", va="top", color=TEXT, fontsize=7,
                bbox=dict(boxstyle="round,pad=0.25", facecolor=PANEL, edgecolor=GRID, alpha=0.85))
    fig.suptitle(f"Phase 1 — physical properties of {len(passed):,} peptides that passed ({100*len(passed)/total:.1f}% of input)", color=TEXT, fontsize=11, y=1.01)
    fig.tight_layout()
    run.log({"phases/plots/phase1_feature_distributions": wandb.Image(_fig_to_pil(fig))})
    plt.close(fig)

    if "organism" in df.columns and df["organism"].notna().any():
        org_stats = (
            df.groupby("organism")
            .agg(n=("sequence", "count"), p1_rate=("phase1_pass", "mean"))
            .reset_index()
        )
        m = masks["phase1-2-3-4"]
        org_pass = df[m].groupby("organism").size()
        org_stats["final_count"] = org_stats["organism"].map(org_pass).fillna(0).astype(int)
        org_stats["final_rate"] = (org_stats["final_count"] / org_stats["n"] * 100).round(1)
        org_stats["p1_rate"] = (org_stats["p1_rate"] * 100).round(1)
        run.log({"phases/tables/organism_breakdown": wandb.Table(dataframe=org_stats)})

        fig, ax = plt.subplots(figsize=(10, max(4, len(org_stats)*0.4)), facecolor=BG)
        y = range(len(org_stats))
        bars_org = ax.barh(y, org_stats["final_rate"], color=PASS_C, alpha=0.85)
        ax.set_yticks(y)
        ax.set_yticklabels(org_stats["organism"], fontsize=8)
        _style_ax(ax, "Final pipeline pass rate by organism (Phases 1–4)", xlabel="% of that organism\u2019s peptides passing")
        for bar, row in zip(bars_org, org_stats.itertuples()):
            ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                    f"{row.final_rate:.1f}% ({int(row.final_count):,}/{int(row.n):,})",
                    va="center", color=TEXT, fontsize=7)
        ax.xaxis.set_major_formatter(mtick.PercentFormatter())
        fig.tight_layout()
        run.log({"phases/plots/organism_pass_rates": wandb.Image(_fig_to_pil(fig))})
        plt.close(fig)

    result_cols = [c for c in [
        "sequence", "organism", "llamp_mic_ug_ml",
        "phase1_pass", "phase2_pass", "phase3_pass", "phase4_pass",
        "phase1_length", "phase1_charge", "phase1_hydrophobic_frac", "phase1_hydrophobic_moment",
        "phase2_blast_identity", "phase2_entropy", "phase2_score",
        "phase3_score", "phase4_score",
    ] if c in df.columns]
    sample = df[result_cols].sample(min(5000, len(df)), random_state=42)
    run.log({"phases/tables/master_results": wandb.Table(dataframe=sample.reset_index(drop=True))})

def run_llamp_cross_org(sequences, home_org, organisms, base_name, resume=True):
    home_org = resolve_organism(home_org)
    if organisms is None:
        organisms = TRAINED_ORGANISMS
    else:
        organisms = [resolve_organism(o) for o in organisms]

    results = {}
    for target_org in organisms:
        safe = target_org.replace(" ", "_")
        ckpt = os.path.join(OUTPUT_DIR, f"{base_name}__{safe}.csv")
        if resume and os.path.exists(ckpt):
            print(f"  [skip] {target_org}: loading checkpoint")
            results[target_org] = pd.read_csv(ckpt)
            continue
        print(f"  [run]  {target_org} ({len(sequences)} sequences) ...")
        df = _run_llamp([(s, target_org) for s in sequences])
        df.to_csv(ckpt, index=False)
        results[target_org] = df
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return results

def compute_llamp_competition(per_org_results, home_org):
    home_org = resolve_organism(home_org)
    organisms = list(per_org_results.keys())
    deduped = {o: df.drop_duplicates(subset="sequence", keep="first") for o, df in per_org_results.items()}

    wide = deduped[organisms[0]][["sequence"]].copy()
    wide["generated_for_organism"] = home_org
    for org in organisms:
        wide = wide.merge(
            deduped[org][["sequence", "llamp_mic_ug_ml"]].rename(columns={"llamp_mic_ug_ml": f"mic_{org}"}),
            on="sequence", how="left",
        )

    mic_cols = [f"mic_{o}" for o in organisms]
    mic_matrix = wide[mic_cols].to_numpy(dtype=float)
    has_valid = ~np.isnan(mic_matrix).all(axis=1)
    row_min = np.nanmin(mic_matrix, axis=1)
    is_winner = np.isclose(mic_matrix, row_min[:, None], atol=1e-9, equal_nan=False)
    is_winner[np.isnan(mic_matrix)] = False
    is_winner[~has_valid, :] = False

    home_col = f"mic_{home_org}"
    home_idx = mic_cols.index(home_col)
    home_is_winner = is_winner[:, home_idx]

    win_counts = {
        "home_wins": int(home_is_winner.sum()),
        "other_wins": {},
        "ties_or_na": int((~has_valid).sum()),
    }
    for j, org in enumerate(organisms):
        if org == home_org:
            continue
        cnt = int((is_winner[:, j] & ~home_is_winner).sum())
        if cnt:
            win_counts["other_wins"][org] = cnt
    return wide, win_counts, organisms, mic_cols

def save_integrated_xlsx(wide, win_counts, home_org, organisms, out_path):
    from openpyxl import Workbook
    from openpyxl.styles import Font
    from openpyxl.utils import get_column_letter

    GREEN = Font(color="008000", bold=True)
    RED = Font(color="CC0000", bold=True)
    mic_cols = [f"mic_{o}" for o in organisms]
    home_col = f"mic_{home_org}"
    mic_matrix = wide[mic_cols].to_numpy(dtype=float)
    has_valid = ~np.isnan(mic_matrix).all(axis=1)
    row_min = np.nanmin(mic_matrix, axis=1)
    is_winner = np.isclose(mic_matrix, row_min[:, None], atol=1e-9, equal_nan=False)
    is_winner[np.isnan(mic_matrix)] = False
    home_idx = mic_cols.index(home_col)
    home_is_winner = is_winner[:, home_idx]

    wb = Workbook()
    ws = wb.active
    ws.title = "integrated_mic_report"
    headers = ["sequence", "generated_for_organism"] + mic_cols
    ws.append(headers)
    for i in range(len(wide)):
        ws.append([wide.iloc[i]["sequence"], wide.iloc[i]["generated_for_organism"]] + mic_matrix[i].tolist())
        excel_row = i + 2
        for j, col_name in enumerate(mic_cols):
            if is_winner[i, j]:
                ws.cell(row=excel_row, column=3 + j).font = GREEN
            elif col_name == home_col and has_valid[i] and not home_is_winner[i]:
                ws.cell(row=excel_row, column=3 + j).font = RED
    for i in range(1, len(headers) + 1):
        ws.column_dimensions[get_column_letter(i)].width = 18
    os.makedirs(os.path.dirname(os.path.abspath(out_path)), exist_ok=True)
    wb.save(out_path)
    print(f"Saved integrated report → {out_path}")

def log_llamp_dashboard(run, per_org_results, home_org, sequences_used, base_name):
    home_org = resolve_organism(home_org)
    wide, win_counts, organisms, mic_cols = compute_llamp_competition(per_org_results, home_org)
    win_stats = compute_llamp_win_stats(wide, organisms, home_org)
    stats_df = win_stats["stats_df"]

    xlsx_path = os.path.join(OUTPUT_DIR, f"{base_name}_integrated_report.xlsx")
    save_integrated_xlsx(wide, win_counts, home_org, organisms, xlsx_path)

    global _LLAMP_DASHBOARD_HTML
    llamp_html = build_llamp_explorer_html(wide, organisms, home_org)
    _LLAMP_DASHBOARD_HTML = llamp_html
    llamp_html_path = os.path.join(OUTPUT_DIR, f"{base_name}_llamp_interactive_dashboard.html")
    with open(llamp_html_path, "w") as f:
        f.write(llamp_html)
    run.log({"llamp/tables/win_rate_percentages": wandb.Table(dataframe=stats_df)})
    run.log({"llamp/artifacts/integrated_xlsx": xlsx_path})

    total = len(wide)
    home_wins = win_counts["home_wins"]
    HOME_COLOR, COMP_COLOR = "#1D9E75", "#888780"

    _log_summary(run, {
        "llamp/home_organism": home_org,
        "llamp/n_sequences": len(sequences_used),
        "llamp/n_organisms_evaluated": len(organisms),
        "llamp/home_wins": home_wins,
        "llamp/home_win_rate_pct": round(100 * home_wins / total, 2) if total else 0,
        "llamp/ties_or_na": win_counts["ties_or_na"],
        **{f"llamp/wins_by_{org.replace(' ', '_')}": c for org, c in win_counts["other_wins"].items()},
    })

    short = lambda o: f"{o.split()[0][0]}. {o.split()[-1]}" if " " in o else o

    fig2, ax2 = plt.subplots(figsize=(10, max(4, len(organisms) * 0.55)), facecolor=BG)
    ax2.barh(range(len(organisms)), stats_df["win_rate_pct"], color=[
        HOME_COLOR if o == home_org else COMP_COLOR for o in stats_df["organism"]
    ], alpha=0.85)
    ax2.set_yticks(range(len(organisms)))
    ax2.set_yticklabels([
        f"★ {short(o)}" if o == home_org else short(o) for o in stats_df["organism"]
    ], fontsize=8)
    ax2.invert_yaxis()
    _style_ax(ax2,
              f"Which organism has the lowest predicted MIC? (% of {total:,} peptides - target = {short(home_org)})",
              xlabel="% of peptides where this organism wins")
    ax2.xaxis.set_major_formatter(mtick.PercentFormatter())
    for i, row in enumerate(stats_df.itertuples()):
        ax2.text(row.win_rate_pct + 0.3, i,
                 f"{row.win_rate_pct:.1f}% ({int(row.win_count):,} wins · {row.win_rate_of_evaluable_pct:.1f}% of evaluable)",
                 va="center", color=TEXT, fontsize=7)
    fig2.tight_layout()
    run.log({"llamp/plots/mic_winner_rates": wandb.Image(_fig_to_pil(fig2))})
    plt.close(fig2)

    evaluable = win_stats["evaluable"]
    if evaluable > 0:
        pie_vals = stats_df["win_count"].values
        pie_labels = [short(o) for o in stats_df["organism"]]
        pie_colors = [HOME_COLOR if o == home_org else COMP_COLOR for o in stats_df["organism"]]
        fig3, ax3 = plt.subplots(figsize=(8, 8), facecolor=BG)
        wedges, texts, autotexts = ax3.pie(
            pie_vals, labels=pie_labels, autopct=lambda p: f"{p:.1f}%",
            colors=pie_colors, textprops={"color": TEXT, "fontsize": 8},
            wedgeprops={"edgecolor": PANEL, "linewidth": 1},
        )
        for t in autotexts:
            t.set_fontsize(7)
        ax3.set_title(f"Share of MIC wins among evaluable peptides (n={evaluable:,}, {100*evaluable/total:.1f}% of input)", color=TEXT, fontsize=10)
        fig3.tight_layout()
        run.log({"llamp/plots/win_share_pie": wandb.Image(_fig_to_pil(fig3))})
        plt.close(fig3)

    medians = []
    for org in organisms:
        vals = pd.to_numeric(wide[f"mic_{org}"], errors="coerce").dropna()
        medians.append(float(np.median(vals)) if len(vals) else np.nan)
    fig, ax = plt.subplots(figsize=(12, 5), facecolor=BG)
    colors = [HOME_COLOR if o == home_org else "#4C9AFF" for o in organisms]
    ax.bar(range(len(organisms)), medians, color=colors, edgecolor=PANEL)
    ax.set_xticks(range(len(organisms)))
    ax.set_xticklabels([short(o) for o in organisms], rotation=30, ha="right", fontsize=8)
    _style_ax(ax, f"Median predicted MIC by organism (lower = stronger · ★ home = {short(home_org)})",
              ylabel="Median MIC (µg/mL)")
    for i, (med, org) in enumerate(zip(medians, organisms)):
        if not np.isnan(med):
            vals = pd.to_numeric(wide[f"mic_{org}"], errors="coerce").dropna()
            pct_valid = 100 * len(vals) / total if total else 0
            ax.text(i, med, f"{med:.1f}\n({pct_valid:.0f}% valid)", ha="center", va="bottom",
                    color=TEXT, fontsize=7)
    fig.tight_layout()
    run.log({"llamp/plots/median_mic_by_organism": wandb.Image(_fig_to_pil(fig))})
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(max(8, len(organisms) * 1.1), 5.5), facecolor=BG)
    violin_data, violin_pos, violin_colors, violin_labels = [], [], [], []
    for j, org in enumerate(organisms):
        vals = pd.to_numeric(wide[f"mic_{org}"], errors="coerce").dropna().values
        if len(vals) >= 2:
            violin_data.append(np.log10(vals + 1e-6))
            violin_pos.append(j)
            violin_colors.append(HOME_COLOR if org == home_org else COMP_COLOR)
            pct_valid = 100 * len(vals) / total if total else 0
            lbl = short(org) + (" ★" if org == home_org else "")
            violin_labels.append(f"{lbl}\n(n={len(vals):,}, {pct_valid:.1f}%)")
        else:
            violin_labels.append(short(org) + (" ★" if org == home_org else ""))
    if violin_data:
        parts = ax.violinplot(violin_data, positions=violin_pos, showmedians=True, widths=0.7)
        for pc, color in zip(parts["bodies"], violin_colors):
            pc.set_facecolor(color)
            pc.set_alpha(0.5)
            pc.set_edgecolor(color)
        parts["cmedians"].set_color("#ffffff")
    ax.set_xticks(range(len(organisms)))
    ax.set_xticklabels(violin_labels, rotation=30, ha="right", fontsize=7)
    for tick, org in zip(ax.get_xticklabels(), organisms):
        tick.set_color(HOME_COLOR if org == home_org else MUTED)
    _style_ax(ax, f"MIC distribution spread by organism (log₁₀ µg/mL · lower = stronger · ★ = {short(home_org)})",
              ylabel="log₁₀(MIC µg/mL)")
    fig.tight_layout()
    run.log({"llamp/plots/mic_violin": wandb.Image(_fig_to_pil(fig))})
    plt.close(fig)

    n = len(organisms)
    ncols, nrows = 3, math.ceil(n / 3)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), facecolor=BG)
    axes = np.array(axes).reshape(-1)
    for i, org in enumerate(organisms):
        ax = axes[i]
        ax.set_facecolor(PANEL)
        mic_vals = per_org_results[org]["llamp_mic_ug_ml"].dropna()
        n_valid = len(mic_vals)
        pct_valid = 100 * n_valid / total if total else 0
        if n_valid:
            ax.hist(mic_vals, bins=20, color=HOME_COLOR if org == home_org else "#4C9AFF", edgecolor=PANEL)
        title = short(org) + (" ★" if org == home_org else "")
        ax.set_title(f"{title}\n{n_valid:,} preds ({pct_valid:.1f}%)",
                     color=HOME_COLOR if org == home_org else TEXT, fontsize=8)
    for j in range(n, len(axes)):
        axes[j].set_visible(False)
    fig.suptitle("Predicted MIC distribution per organism (µg/mL — lower is better)", color=TEXT, fontsize=11)
    fig.tight_layout()
    run.log({"llamp/plots/mic_grid": wandb.Image(_fig_to_pil(fig))})
    plt.close(fig)

    table_rows = []
    for org in organisms:
        wins = home_wins if org == home_org else win_counts["other_wins"].get(org, 0)
        vals = pd.to_numeric(wide[f"mic_{org}"], errors="coerce").dropna()
        med = f"{np.median(vals):.2f}" if len(vals) else "—"
        table_rows.append([
            org, "HOME (target)" if org == home_org else "competitor",
            wins, f"{100 * wins / total:.1f}%" if total else "—", med,
        ])
    run.log({"llamp/tables/organism_competition": wandb.Table(
        columns=["Organism", "Role", "Win Count", "Win Rate", "Median MIC (µg/mL)"],
        data=table_rows,
    )})

    for org, odf in per_org_results.items():
        safe = org.replace(" ", "_")
        sample = odf.sample(min(2000, len(odf)), random_state=42)
        run.log({f"llamp/tables/results_{safe}": wandb.Table(dataframe=sample.reset_index(drop=True))})

    return wide, win_counts

print("Helper functions loaded. Run the Target Evaluation cell next.")


## Target evaluation runner

Edit `TARGET_INPUTS`, then run to evaluate one CSV per target organism.


In [ ]:
import base64, gc, json, os, re
from datetime import datetime

TARGET_INPUTS = {
     "subtilis":   "/content/drive/MyDrive/DBAASP_Bioinformatics/generatedPeptides/B_subtilis.csv",
}

TARGET_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "target_evaluation")
TARGET_RUN_NAME   = f"target_evaluation_{datetime.now():%Y%m%d_%H%M}"
TARGET_RUN_PHASES = True
TARGET_RUN_LLAMP  = False
TARGET_LLAMP_ON   = "filtered"
TARGET_LLAMP_ORGANISMS = LLAMP_ORGANISMS or TRAINED_ORGANISMS
TARGET_LLAMP_RESUME    = False

def _require_target_helpers():
    needed = [
        "load_input_df", "run_all_phases_once", "build_combination_summary",
        "build_pipeline_explorer_html", "resolve_organism", "run_llamp_cross_org",
        "compute_llamp_competition", "compute_llamp_win_stats", "build_llamp_explorer_html",
        "save_integrated_xlsx", "wandb",
    ]
    missing = [name for name in needed if name not in globals()]
    if missing:
        raise RuntimeError(
            "Run the Universal Evaluation Runner cell once first so these helpers exist: "
            + ", ".join(missing)
        )

def _safe_key(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text)).strip("_") or "home"

def _normalize_target_inputs(items):
    if isinstance(items, dict):
        pairs = list(items.items())
    else:
        pairs = list(items)
    if not pairs:
        raise ValueError("TARGET_INPUTS is empty. Add at least one target -> CSV path.")
    normalized = []
    for label, path in pairs:
        if not path or str(path).startswith("path/to/"):
            raise ValueError(f"Replace the placeholder path for target '{label}'.")
        home = resolve_organism(label)
        normalized.append({"label": str(label), "home": home, "path": str(path), "key": _safe_key(home)})
    return normalized

def _b64_html(html_text):
    return base64.b64encode(html_text.encode("utf-8")).decode("ascii")

def build_target_dashboard_html(target_payloads):
    data_json = json.dumps(target_payloads)
    options = "\n".join(
        f'<option value="{i}">{p["target"]} ({p["n_input"]:,} peptides)</option>'
        for i, p in enumerate(target_payloads)
    )
    return f"""<!DOCTYPE html>
<html><head><meta charset="utf-8">
<style>
  :root {{ --bg:#1a1a1a; --panel:#242424; --grid:#333; --text:#d4d2c8; --muted:#888780;
           --pass:#1D9E75; --fail:#E24B4A; --accent:#4C9AFF; --home:#E8A838; }}
  body {{ font-family:'Segoe UI',system-ui,sans-serif; background:var(--bg); color:var(--text);
          padding:22px; max-width:1280px; margin:0 auto; }}
  h1 {{ margin:0 0 5px; font-size:1.55rem; }}
  h2 {{ color:var(--muted); font-size:.82rem; letter-spacing:.08em; text-transform:uppercase;
        border-bottom:1px solid var(--grid); padding-bottom:6px; margin:22px 0 12px; }}
  .sub {{ color:var(--muted); margin:0 0 16px; }}
  .chooser {{ display:flex; gap:12px; align-items:center; background:var(--panel); border:1px solid var(--grid);
              border-radius:12px; padding:14px; position:sticky; top:0; z-index:5; }}
  select {{ background:#111; color:var(--text); border:1px solid var(--grid); border-radius:8px;
            padding:9px 12px; min-width:340px; font-size:.95rem; }}
  .cards {{ display:grid; grid-template-columns:repeat(5,1fr); gap:10px; margin:14px 0; }}
  @media(max-width:900px){{ .cards {{ grid-template-columns:repeat(2,1fr); }} .chooser {{ flex-wrap:wrap; }} }}
  .card {{ background:var(--panel); border:1px solid var(--grid); border-radius:10px; padding:12px; text-align:center; }}
  .card .val {{ font-size:1.35rem; font-weight:800; color:var(--pass); }}
  .card .lbl {{ color:var(--muted); font-size:.68rem; margin-top:4px; line-height:1.25; }}
  .homeval {{ color:var(--home)!important; }}
  .integrated {{ background:var(--panel); border:1px solid var(--grid); border-radius:12px; padding:14px; }}
  table {{ width:100%; border-collapse:collapse; }}
  th,td {{ padding:8px 9px; border-bottom:1px solid var(--grid); font-size:.82rem; text-align:left; }}
  th {{ color:var(--muted); text-transform:uppercase; font-size:.7rem; }}
  tr.active {{ background:#2a2418; }}
  .bar-bg {{ height:8px; background:#111; border-radius:999px; overflow:hidden; min-width:110px; }}
  .bar {{ height:100%; background:var(--accent); border-radius:999px; }}
  .bar.home {{ background:var(--home); }}
  .tabs {{ display:flex; gap:8px; margin:16px 0 10px; }}
  .tab {{ background:var(--panel); color:var(--text); border:1px solid var(--grid); border-radius:8px;
          padding:8px 12px; cursor:pointer; }}
  .tab.on {{ border-color:var(--pass); color:var(--pass); }}
  iframe {{ width:100%; height:960px; border:1px solid var(--grid); border-radius:12px; background:var(--bg); }}
  .note {{ color:var(--muted); font-size:.78rem; margin-top:8px; }}
</style></head><body>
<h1>Evaluation Dashboard</h1>
<p class="sub">Select a target to view its generated peptides, phase filters, and LLAMP organism comparison.</p>
<div class="chooser">
  <strong>Target</strong>
  <select id="homeSelect">{options}</select>
  <span id="pathText" class="sub"></span>
</div>
<div class="cards">
  <div class="card"><div class="val" id="mInput">-</div><div class="lbl">Input peptides</div></div>
  <div class="card"><div class="val" id="mPhase">-</div><div class="lbl">Pass P1+P2+P3+P4</div></div>
  <div class="card"><div class="val" id="mPhasePct">-</div><div class="lbl">Phase retention</div></div>
  <div class="card"><div class="val homeval" id="mHomeWin">-</div><div class="lbl">Target LLAMP win rate</div></div>
  <div class="card"><div class="val" id="mLeader">-</div><div class="lbl">Top target by win rate</div></div>
</div>
<h2>Integrated Summary</h2>
<div class="integrated">
  <table><thead><tr><th>Target</th><th>Input</th><th>Phase survivors</th><th>Retention</th><th>LLAMP peptides</th><th>Target wins</th><th>Win rate</th><th>Visual</th></tr></thead><tbody id="summaryRows"></tbody></table>
  <div class="note">Win rate is computed on the LLAMP evaluation set for each target. When TARGET_LLAMP_ON = "filtered", that set is the phase survivors.</div>
</div>
<div class="tabs">
  <button class="tab on" id="tabPhases">Phases Dashboard</button>
  <button class="tab" id="tabLlamp">LLAMP Dashboard</button>
</div>
<iframe id="dashFrame" sandbox="allow-scripts allow-same-origin"></iframe>
<script>
const TARGETS={data_json};
let mode='phases';
const shortName=(s)=>s ? s.split(' ').map((x,i)=>i===0?x[0]+'.':x).join(' ') : '-';
function pct(v){{ return v===null || isNaN(v) ? '-' : Number(v).toFixed(1)+'%'; }}
function bestHome(){{
  const ranked=TARGETS.filter(h=>h.target_win_rate_pct!==null).sort((a,b)=>b.target_win_rate_pct-a.target_win_rate_pct);
  return ranked.length ? shortName(ranked[0].target) : '-';
}}
function rows(activeIdx){{
  const max=Math.max(...TARGETS.map(h=>h.target_win_rate_pct||0), 1);
  summaryRows.innerHTML=TARGETS.map((h,i)=>{{
    const rate=h.target_win_rate_pct;
    const w=Math.round(100*(rate||0)/max);
    return `<tr class="${{i===activeIdx?'active':''}}"><td>${{h.target}}</td><td>${{h.n_input.toLocaleString()}}</td>
      <td>${{h.phase_final_pass===null?'-':h.phase_final_pass.toLocaleString()}}</td><td>${{pct(h.phase_retention_pct)}}</td>
      <td>${{h.llamp_n_sequences===null?'-':h.llamp_n_sequences.toLocaleString()}}</td>
      <td>${{h.target_wins===null?'-':h.target_wins.toLocaleString()}}</td><td>${{pct(rate)}}</td>
      <td><div class="bar-bg"><div class="bar ${{i===activeIdx?'home':''}}" style="width:${{w}}%"></div></div></td></tr>`;
  }}).join('');
}}
function decodeHtml(html64){{
  const bytes=Uint8Array.from(atob(html64), c=>c.charCodeAt(0));
  return new TextDecoder('utf-8').decode(bytes);
}}
function update(){{
  const idx=+homeSelect.value;
  const h=TARGETS[idx];
  pathText.textContent=h.path;
  mInput.textContent=h.n_input.toLocaleString();
  mPhase.textContent=h.phase_final_pass===null?'-':h.phase_final_pass.toLocaleString();
  mPhasePct.textContent=pct(h.phase_retention_pct);
  mHomeWin.textContent=pct(h.target_win_rate_pct);
  mLeader.textContent=bestHome();
  rows(idx);
  const html64 = mode==='phases' ? h.phase_html_b64 : h.llamp_html_b64;
  dashFrame.srcdoc = html64 ? decodeHtml(html64) : '<body style="background:#1a1a1a;color:#d4d2c8;font-family:sans-serif;padding:24px">This section was not run for the selected target.</body>';
  tabPhases.classList.toggle('on', mode==='phases');
  tabLlamp.classList.toggle('on', mode==='llamp');
}}
homeSelect.addEventListener('change', update);
tabPhases.addEventListener('click', ()=>{{mode='phases'; update();}});
tabLlamp.addEventListener('click', ()=>{{mode='llamp'; update();}});
update();
</script></body></html>"""

def run_target_evaluation():
    _require_target_helpers()
    homes = _normalize_target_inputs(TARGET_INPUTS)
    os.makedirs(TARGET_OUTPUT_DIR, exist_ok=True)

    run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name=TARGET_RUN_NAME,
        job_type="target_evaluation",
        config={
            "target_inputs": {h["home"]: h["path"] for h in homes},
            "n_targets": len(homes),
            "run_phases": TARGET_RUN_PHASES,
            "run_llamp": TARGET_RUN_LLAMP,
            "llamp_on": TARGET_LLAMP_ON,
            "llamp_organisms": TARGET_LLAMP_ORGANISMS,
            "phase2_threshold": PHASE2_THRESHOLD,
            "phase3_threshold": PHASE3_THRESHOLD,
            "phase4_threshold": PHASE4_THRESHOLD,
        },
    )

    dashboard_payloads = []
    summary_rows = []

    try:
        for i, spec in enumerate(homes, start=1):
            home, path, key = spec["home"], spec["path"], spec["key"]
            base_name = f"{key}_{os.path.splitext(os.path.basename(path))[0]}"
            print(f"\n{'='*76}\n[{i}/{len(homes)}] Target: {home}\nInput: {path}")

            raw_df = load_input_df(path)
            total = len(raw_df)
            phase_df = combo_df = masks = None
            phase_html = ""
            llamp_html = ""
            phase_final_pass = None
            phase_retention_pct = None
            llamp_home_wins = None
            llamp_home_win_rate_pct = None
            llamp_home_win_rate_evaluable_pct = None

            if TARGET_RUN_PHASES:
                phase_df = run_all_phases_once(raw_df.copy())
                combo_df, masks = build_combination_summary(phase_df, total)
                thresholds = {"p2": PHASE2_THRESHOLD, "p3": PHASE3_THRESHOLD, "p4": PHASE4_THRESHOLD}
                phase_html = build_pipeline_explorer_html(phase_df, combo_df, total, thresholds)
                phase_final_pass = int(masks["phase1-2-3-4"].sum())
                phase_retention_pct = round(100 * phase_final_pass / total, 2) if total else 0

                phase_csv = os.path.join(TARGET_OUTPUT_DIR, f"{base_name}_phases_full.csv")
                phase_df.to_csv(phase_csv, index=False)
                run.log({f"target_evaluation/{key}/phases_csv": phase_csv})
                run.log({f"target_evaluation/{key}/phase_summary": wandb.Table(dataframe=combo_df)})
                print(f"✓ Phases complete: {phase_final_pass:,}/{total:,} retained ({phase_retention_pct:.1f}%)")

            llamp_n_sequences = None
            llamp_evaluable = None
            if TARGET_RUN_LLAMP:
                if TARGET_RUN_PHASES and TARGET_LLAMP_ON == "filtered" and masks is not None:
                    seqs = phase_df.loc[masks["phase1-2-3-4"], "sequence"].tolist()
                else:
                    seqs = raw_df["sequence"].tolist()
                llamp_n_sequences = len(seqs)
                print(f"── LLAMP: {len(seqs):,} sequences x {len(TARGET_LLAMP_ORGANISMS)} organisms")

                if not seqs:
                    print("LLAMP skipped for this target because no sequences were selected.")
                    per_org = None
                else:
                    per_org = run_llamp_cross_org(
                        sequences=seqs,
                        home_org=home,
                        organisms=TARGET_LLAMP_ORGANISMS,
                        base_name=base_name,
                        resume=TARGET_LLAMP_RESUME,
                    )
                if per_org is not None:
                    wide, win_counts, organisms, _ = compute_llamp_competition(per_org, home)
                    win_stats = compute_llamp_win_stats(wide, organisms, home)
                    llamp_evaluable = int(win_stats["evaluable"])
                    llamp_html = build_llamp_explorer_html(wide, organisms, home)
                    llamp_home_wins = int(win_counts["home_wins"])
                    llamp_home_win_rate_pct = round(100 * llamp_home_wins / len(wide), 2) if len(wide) else 0
                    llamp_home_win_rate_evaluable_pct = round(100 * llamp_home_wins / llamp_evaluable, 2) if llamp_evaluable else 0

                    xlsx_path = os.path.join(TARGET_OUTPUT_DIR, f"{base_name}_integrated_report.xlsx")
                    save_integrated_xlsx(wide, win_counts, home, organisms, xlsx_path)
                    run.log({f"target_evaluation/{key}/llamp_win_stats": wandb.Table(dataframe=win_stats["stats_df"])})
                    run.log({f"target_evaluation/{key}/llamp_integrated_xlsx": xlsx_path})
                    print(f"LLAMP complete: target wins {llamp_home_wins:,}/{len(wide):,} ({llamp_home_win_rate_pct:.1f}% of LLAMP set; {llamp_home_win_rate_evaluable_pct:.1f}% of evaluable)")

            payload = {
                "target": home,
                "home": home,
                "label": spec["label"],
                "path": path,
                "key": key,
                "n_input": total,
                "phase_final_pass": phase_final_pass,
                "phase_retention_pct": phase_retention_pct,
                "llamp_n_sequences": llamp_n_sequences,
                "llamp_evaluable": llamp_evaluable,
                "target_wins": llamp_home_wins,
                "target_win_rate_pct": llamp_home_win_rate_pct,
                "target_win_rate_evaluable_pct": llamp_home_win_rate_evaluable_pct,
                "phase_html_b64": _b64_html(phase_html) if phase_html else "",
                "llamp_html_b64": _b64_html(llamp_html) if llamp_html else "",
            }
            dashboard_payloads.append(payload)
            summary_rows.append({k: payload[k] for k in [
                "target", "path", "n_input", "phase_final_pass", "phase_retention_pct",
                "llamp_n_sequences", "llamp_evaluable", "target_wins",
                "target_win_rate_pct", "target_win_rate_evaluable_pct",
            ]})

        summary_df = pd.DataFrame(summary_rows)
        summary_csv = os.path.join(TARGET_OUTPUT_DIR, "target_evaluation_summary.csv")
        summary_df.to_csv(summary_csv, index=False)

        combined_html = build_target_dashboard_html(dashboard_payloads)
        combined_html_path = os.path.join(TARGET_OUTPUT_DIR, "evaluation_dashboard.html")
        with open(combined_html_path, "w") as f:
            f.write(combined_html)

        run.log({"dashboards/evaluation_dashboard": wandb.Html(combined_html, inject=False)})
        run.log({"target_evaluation/summary": wandb.Table(dataframe=summary_df)})
        run.log({"target_evaluation/summary_csv": summary_csv})
        run.summary.update({
            "target_evaluation/n_targets": len(homes),
            "target_evaluation/dashboard_path": combined_html_path,
        })

        print(f"\n{'='*76}")
        print("Target evaluation complete")
        print("  W&B dashboard key: dashboards/evaluation_dashboard")
        print(f"  Local dashboard: {combined_html_path}")
        print(f"  Summary CSV: {summary_csv}")
        print(f"  https://wandb.ai/{WANDB_ENTITY}/{WANDB_PROJECT}/runs/{run.id}")
        print(f"{'='*76}")
        return summary_df

    finally:
        wandb.finish()
        plt.close("all")
        gc.collect()

target_summary_df = run_target_evaluation()
target_summary_df
